In [12]:
import pandas as pd

df = pd.read_csv('NBA_Global_Media_Insights/Data/raw/nba_2025_26_schedule.csv')

In [14]:
# Keep regular-season games
df = df[df["season_type"] == "Regular Season"].copy()

# Remove games where teams are not yet known
df = df[df["is_tbd"] == False].copy()

# Keep nationally televised games
df = df[df["national_tv"].notna()].copy()

print("Nationally televised games:", len(df))
print(df["national_tv"].value_counts())

# Save processed data
df.to_csv(
    "NBA_Global_Media_Insights/Data/processed/nba_national_telecasts_2025_26.csv",
    index=False
)

Nationally televised games: 297
Prime Video            60
NBA TV                 60
NBC | Peacock          58
ESPN                   55
Peacock                40
ABC                    18
ABC | ESPN              5
Telemundo | Peacock     1
Name: national_tv, dtype: int64


In [16]:
telecasts = pd.read_csv('NBA_Global_Media_Insights/Data/processed/nba_national_telecasts_2025_26.csv')

print(telecasts.shape)
telecasts.head()

(297, 14)


,game_id,season_type,tip_utc,tip_et,away_team,home_team,venue,city,state,national_tv,home_rsns,away_rsns,international,is_tbd
0,22500001,Regular Season,2025-10-21T23:30:00Z,2025-10-21T18:30:00-05:00,HOU (Rockets),OKC (Thunder),Paycom Center,Oklahoma City,OK,NBC | Peacock,NaN,NaN,NaN,False
1,22500002,Regular Season,2025-10-22T02:00:00Z,2025-10-21T21:00:00-05:00,GSW (Warriors),LAL (Lakers),Crypto.com Arena,Los Angeles,CA,NBC | Peacock,NaN,NaN,NaN,False
2,22500003,Regular Season,2025-10-22T23:00:00Z,2025-10-22T18:00:00-05:00,CLE (Cavaliers),NYK (Knicks),Madison Square Garden,New York,NY,ESPN,MSG,NaN,NaN,False
3,22500004,Regular Season,2025-10-23T01:30:00Z,2025-10-22T20:30:00-05:00,SAS (Spurs),DAL (Mavericks),American Airlines Center,Dallas,TX,ESPN,NaN,NaN,NaN,False
4,22500005,Regular Season,2025-10-23T23:30:00Z,2025-10-23T18:30:00-05:00,OKC (Thunder),IND (Pacers),Gainbridge Fieldhouse,Indianapolis,IN,ESPN,FanDuel Sports Network - Indiana,FanDuel Sports Network - Oklahoma,NaN,False


In [17]:
telecasts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 297 entries, 0 to 296
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   game_id        297 non-null    int64  
 1   season_type    297 non-null    object 
 2   tip_utc        297 non-null    object 
 3   tip_et         297 non-null    object 
 4   away_team      297 non-null    object 
 5   home_team      297 non-null    object 
 6   venue          297 non-null    object 
 7   city           297 non-null    object 
 8   state          295 non-null    object 
 9   national_tv    297 non-null    object 
 10  home_rsns      158 non-null    object 
 11  away_rsns      172 non-null    object 
 12  international  0 non-null      float64
 13  is_tbd         297 non-null    bool   
dtypes: bool(1), float64(1), int64(1), object(11)
memory usage: 30.6+ KB


In [18]:
telecasts.isna().sum()

game_id            0
season_type        0
tip_utc            0
tip_et             0
away_team          0
home_team          0
venue              0
city               0
state              2
national_tv        0
home_rsns        139
away_rsns        125
international    297
is_tbd             0
dtype: int64

In [21]:
telecasts['national_tv'].value_counts()

Prime Video            60
NBA TV                 60
NBC | Peacock          58
ESPN                   55
Peacock                40
ABC                    18
ABC | ESPN              5
Telemundo | Peacock     1
Name: national_tv, dtype: int64

In [22]:
telecasts[['game_id', 'tip_et', 'away_team', 'home_team', 'national_tv']].head(20)

,game_id,tip_et,away_team,home_team,national_tv
0,22500001,2025-10-21T18:30:00-05:00,HOU (Rockets),OKC (Thunder),NBC | Peacock
1,22500002,2025-10-21T21:00:00-05:00,GSW (Warriors),LAL (Lakers),NBC | Peacock
2,22500003,2025-10-22T18:00:00-05:00,CLE (Cavaliers),NYK (Knicks),ESPN
3,22500004,2025-10-22T20:30:00-05:00,SAS (Spurs),DAL (Mavericks),ESPN
4,22500005,2025-10-23T18:30:00-05:00,OKC (Thunder),IND (Pacers),ESPN
5,22500006,2025-10-23T21:00:00-05:00,DEN (Nuggets),GSW (Warriors),ESPN
6,22500018,2025-10-24T18:30:00-05:00,BOS (Celtics),NYK (Knicks),Prime Video
7,22500019,2025-10-24T21:00:00-05:00,MIN (Timberwolves),LAL (Lakers),Prime Video
8,22500101,2025-10-25T18:30:00-05:00,OKC (Thunder),ATL (Hawks),NBA TV
9,22500007,2025-10-27T18:00:00-05:00,CLE (Cavaliers),DET (Pistons),Peacock


In [23]:
print(telecasts['tip_et'].min())
print(telecasts['tip_et'].max())

2025-10-21T18:30:00-05:00
2026-04-10T21:00:00-05:00


In [24]:
print("Duplicate game IDs:", telecasts['game_id'].duplicated().sum())

Duplicate game IDs: 0


In [1]:
!pip install requests feedparser

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 1.4 MB/s eta 0:00:00a 0:00:01


In [2]:
import requests
import feedparser

url = "https://ratingsreference.com/feed/nba.xml"

response = requests.get(url)

print(response.status_code)
print(response.headers.get("content-type"))
print(response.text[:500])

200
application/rss+xml; charset=utf-8
<?xml version="1.0" encoding="UTF-8"?>
<rss version="2.0" xmlns:atom="http://www.w3.org/2005/Atom">
<channel>
<title>RatingsReference — newly published NBA ratings</title>
<link>https://ratingsreference.com/league/nba</link>
<description>Audience figures newly published or revised for NBA telecasts, with source and measurement provider. Newest first.</description>
<atom:link href="https://ratingsreference.com/feed/nba.xml" rel="self" type="application/rss+xml"/>
<lastBuildDate>Sun, 27 Sep 2026 0


In [3]:
feed = feedparser.parse(response.content)

print("Number of entries:", len(feed.entries))

for entry in feed.entries[:10]:
    print(entry)

Number of entries: 50
{'title': 'Philadelphia 76ers at Atlanta Hawks (2021-06-18): 4,340,000 average audience — Nielsen, final', 'title_detail': {'type': 'text/plain', 'language': None, 'base': '', 'value': 'Philadelphia 76ers at Atlanta Hawks (2021-06-18): 4,340,000 average audience — Nielsen, final'}, 'links': [{'rel': 'alternate', 'type': 'text/html', 'href': 'https://ratingsreference.com/telecast/nba-76ers-hawks-2021-06-18'}], 'link': 'https://ratingsreference.com/telecast/nba-76ers-hawks-2021-06-18', 'id': 'c-nba-76ers-hawks-2021-06-18-nielsen-avg_audience-full_telecast-unspecified-4340000', 'guidislink': False, 'published': 'Sun, 27 Sep 2026 09:08:01 GMT', 'published_parsed': time.struct_time(tm_year=2026, tm_mon=9, tm_mday=27, tm_hour=9, tm_min=8, tm_sec=1, tm_wday=6, tm_yday=270, tm_isdst=0), 'summary': 'NBA · ESPN · published by Sports Media Watch', 'summary_detail': {'type': 'text/html', 'language': None, 'base': '', 'value': 'NBA · ESPN · published by Sports Media Watch'}, '

In [5]:
import requests
import pandas as pd
from io import StringIO

url = "https://ratingsreference.com/league/nba?year=2025"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/153.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

response = requests.get(url, headers=headers)

print("Status code:", response.status_code)
print("Content length:", len(response.text))

Status code: 200
Content length: 45407


In [6]:
tables = pd.read_html(StringIO(response.text))

print("Number of tables:", len(tables))

for i, table in enumerate(tables):
    print("\nTABLE", i)
    print(table.head())
    print("Shape:", table.shape)

Number of tables: 1

TABLE 0
  dateidate the telecast's air date. On a series row, the year.Methodology â  \
0                                         2025-12-25                             
1                                         2025-12-25                             
2                                         2025-12-25                             
3                                         2025-12-16                             
4                                         2025-12-09                             

  telecastitelecast the telecast the figure is attached to, with its air date. A figure with no telecast row is listed without one.Methodology â  \
0                             NBA CavaliersâKnicks                                                                                                 
1                               NBA RocketsâLakers                                                                                                 
2          Dallas Mavericks 

In [8]:
import requests
import pandas as pd
from io import StringIO

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/153.0.0.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,application/xml;"
        "q=0.9,image/avif,image/webp,*/*;q=0.8"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

def scrape_ratings_reference(year, page=1):
    
    url = f"https://ratingsreference.com/league/nba?page={page}&year={year}"
    
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    
    # Fix character encoding
    response.encoding = "utf-8"
    
    tables = pd.read_html(StringIO(response.text))
    
    # Ratings Reference has one main table
    df = tables[0].copy()
    
    # Simplify column names
    df.columns = ["date", "telecast", "figure"]
    
    return df

In [9]:
ratings_2025_p1 = scrape_ratings_reference(2025, page=1)

print(ratings_2025_p1.shape)
ratings_2025_p1.head(10)

(101, 3)


,date,telecast,figure
0,2025-12-25,NBA Cavaliers–Knicks,6370000.0
1,2025-12-25,NBA Rockets–Lakers,5400000.0
2,2025-12-25,Dallas Mavericks at Golden State Warriors,6100000.0
3,2025-12-16,San Antonio Spurs at New York Knicks,3070000.0
4,2025-12-09,New York Knicks at Toronto Raptors,1140000.0
5,2025-11-26,Houston Rockets at Golden State Warriors,2031000.0
6,2025-11-26,Minnesota Timberwolves at Oklahoma City Thunder,2433000.0
7,2025-11-14,NBA Warriors–Spurs,1430000.0
8,2025-10-31,NBA Celtics–76ers,961000.0
9,2025-10-31,NBA Lakers–Grizzlies,1200000.0


In [10]:
ratings_2025_p2 = scrape_ratings_reference(2025, page=2)

print(ratings_2025_p2.shape)
ratings_2025_p2.head(10)

(18, 3)


,date,telecast,figure
0,2025-01-28,NBA Lakers–76ers,884000
1,2025-01-25,NBA Lakers–Warriors,3050000
2,2025-01-25,NBA Nuggets–Timberwolves,1540000
3,2025-01-23,NBA Heat–Bucks,1050000
4,2025-01-23,NBA Celtics–Lakers,1200000
5,2025-01-22,NBA Warriors–Kings,1090000
6,2025-01-22,NBA Timberwolves–Mavericks,1170000
7,2025-01-21,NBA 76ers–Nuggets,840000
8,2025-01-21,NBA Knicks–Nets,979000
9,2025-01-17,NBA Magic–Celtics,1150000


In [11]:
print(ratings_2025_p2.tail())

          date                   telecast   figure
13  2025-01-15           NBA Knicks–76ers  1220000
14  2025-01-15  NBA Warriors–Timberwolves  1150000
15  2025-01-14       NBA Cavaliers–Pacers  1030000
16  2025-01-08            NBA Spurs–Bucks  1380000
17  2025-01-08      NBA Thunder–Cavaliers  1870000


In [12]:
ratings_2026_p1 = scrape_ratings_reference(2026, page=1)
ratings_2026_p2 = scrape_ratings_reference(2026, page=2)

print("2026 page 1:", ratings_2026_p1.shape)
print("2026 page 2:", ratings_2026_p2.shape)

2026 page 1: (60, 3)
2026 page 2: (60, 3)


In [13]:
ratings_2026_p1.head()

,date,telecast,figure
0,2026-06-13,NBA Finals Game 5,24540000.0
1,2026-06-10,NBA Finals Game 4,20940000.0
2,2026,NBA Finals - Knicks vs Spurs2 games,NaN
3,2026-06-05,NBA Finals Game 2,16430000.0
4,2026-06-08,NBA Finals Game 3,23790000.0


In [14]:
ratings_2026_p2.head()

,date,telecast,figure
0,2026-06-13,NBA Finals Game 5,24540000.0
1,2026-06-10,NBA Finals Game 4,20940000.0
2,2026,NBA Finals - Knicks vs Spurs2 games,NaN
3,2026-06-05,NBA Finals Game 2,16430000.0
4,2026-06-08,NBA Finals Game 3,23790000.0


In [15]:
print("2026 P1 shape:", ratings_2026_p1.shape)
print("2026 P2 shape:", ratings_2026_p2.shape)

print("\n2026 P1 date range:")
print(ratings_2026_p1["date"].min(), "to", ratings_2026_p1["date"].max())

print("\n2026 P2 date range:")
print(ratings_2026_p2["date"].min(), "to", ratings_2026_p2["date"].max())

print("\nExact duplicate rows between P1 and P2:")
print(
    len(
        ratings_2026_p1.merge(
            ratings_2026_p2,
            on=["date", "telecast", "figure"],
            how="inner"
        )
    )
)

2026 P1 shape: (60, 3)
2026 P2 shape: (60, 3)

2026 P1 date range:
2026 to 2026-06-13

2026 P2 date range:
2026 to 2026-06-13

Exact duplicate rows between P1 and P2:
60


In [16]:
pd.set_option("display.max_rows", 100)

ratings_2026_p1

,date,telecast,figure
0,2026-06-13,NBA Finals Game 5,24540000.0
1,2026-06-10,NBA Finals Game 4,20940000.0
2,2026,NBA Finals - Knicks vs Spurs2 games,NaN
3,2026-06-05,NBA Finals Game 2,16430000.0
4,2026-06-08,NBA Finals Game 3,23790000.0
5,2026-06-03,NBA Finals Game 1,16930000.0
6,2026-05-30,San Antonio Spurs at Oklahoma City Thunder,15900000.0
7,2026-05-28,NBA Thunder–Spurs,11590000.0
8,2026-05-21,NBA Cavaliers–Knicks,7100000.0
9,2026-05-20,Western Conference Finals Game 2,7770000.0


In [17]:
print(ratings_2026_p1.dtypes)
print("\nRows with missing figures:")
print(ratings_2026_p1[ratings_2026_p1["figure"].isna()])

date         object
telecast     object
figure      float64
dtype: object

Rows with missing figures:
   date                             telecast  figure
2  2026  NBA Finals - Knicks vs Spurs2 games     NaN


In [18]:
ratings_2026 = ratings_2026_p1.copy()

# Remove non-game/section rows with no audience figure
ratings_2026 = ratings_2026.dropna(subset=["figure"]).copy()

# Convert dates to proper datetime values
ratings_2026["date"] = pd.to_datetime(
    ratings_2026["date"],
    errors="coerce"
)

# Remove anything that isn't a valid date
ratings_2026 = ratings_2026.dropna(subset=["date"]).copy()

# Convert audience figure to integer
ratings_2026["figure"] = ratings_2026["figure"].astype(int)

# Sort chronologically
ratings_2026 = ratings_2026.sort_values("date").reset_index(drop=True)

print("Rows:", len(ratings_2026))
print("Date range:", ratings_2026["date"].min(), "to", ratings_2026["date"].max())

ratings_2026.head(10)

Rows: 59
Date range: 2026-01-24 00:00:00 to 2026-06-13 00:00:00


,date,telecast,figure
0,2026-01-24,Los Angeles Lakers at Dallas Mavericks,2570000
1,2026-01-31,Dallas Mavericks at Houston Rockets,1430000
2,2026-02-01,NBA Lakers–Knicks,3600000
3,2026-02-01,Oklahoma City Thunder at Denver Nuggets,2290000
4,2026-02-07,Golden State Warriors at Los Angeles Lakers,2470000
5,2026-02-21,Houston Rockets at New York Knicks,1360000
6,2026-02-22,NBA Celtics–Lakers,4520000
7,2026-02-28,Los Angeles Lakers at Golden State Warriors,1460000
8,2026-03-01,Philadelphia 76ers at Boston Celtics,2620000
9,2026-03-04,NBA Thunder–Knicks,1580000


In [22]:
print("2025 P1 shape:", ratings_2025_p1.shape)

print("\n2025 date range:")
print(ratings_2025_p1["date"].min(), "to", ratings_2025_p1["date"].max())

print("\nLast 10 rows:")
display(ratings_2025_p1.head(10))

2025 P1 shape: (101, 3)

2025 date range:
2025 to 2025-12-25

Last 10 rows:


,date,telecast,figure
0,2025-12-25,NBA Cavaliers–Knicks,6370000.0
1,2025-12-25,NBA Rockets–Lakers,5400000.0
2,2025-12-25,Dallas Mavericks at Golden State Warriors,6100000.0
3,2025-12-16,San Antonio Spurs at New York Knicks,3070000.0
4,2025-12-09,New York Knicks at Toronto Raptors,1140000.0
5,2025-11-26,Houston Rockets at Golden State Warriors,2031000.0
6,2025-11-26,Minnesota Timberwolves at Oklahoma City Thunder,2433000.0
7,2025-11-14,NBA Warriors–Spurs,1430000.0
8,2025-10-31,NBA Celtics–76ers,961000.0
9,2025-10-31,NBA Lakers–Grizzlies,1200000.0


In [23]:
ratings_2025 = ratings_2025_p1.copy()

# Convert date to datetime
ratings_2025["date"] = pd.to_datetime(
    ratings_2025["date"],
    errors="coerce"
)

# Keep only the 2025-26 NBA season portion
ratings_2025 = ratings_2025[
    ratings_2025["date"] >= "2025-10-01"
].copy()

# Remove rows without an audience figure
ratings_2025 = ratings_2025.dropna(subset=["figure"]).copy()

# Convert audience figure to integer
ratings_2025["figure"] = ratings_2025["figure"].astype(int)

# Sort chronologically
ratings_2025 = ratings_2025.sort_values("date").reset_index(drop=True)

print("Rows:", len(ratings_2025))
print("Date range:", ratings_2025["date"].min(), "to", ratings_2025["date"].max())

display(ratings_2025)

Rows: 15
Date range: 2025-10-21 00:00:00 to 2025-12-25 00:00:00


,date,telecast,figure
0,2025-10-21,NBA Rockets–Thunder,5900000
1,2025-10-22,NBA Cavaliers–Knicks,2170000
2,2025-10-22,NBA Spurs–Mavericks,2490000
3,2025-10-29,NBA Cavaliers–Celtics,1280000
4,2025-10-29,NBA Lakers–Timberwolves,1360000
5,2025-10-31,NBA Celtics–76ers,961000
6,2025-10-31,NBA Lakers–Grizzlies,1200000
7,2025-11-14,NBA Warriors–Spurs,1430000
8,2025-11-26,Houston Rockets at Golden State Warriors,2031000
9,2025-11-26,Minnesota Timberwolves at Oklahoma City Thunder,2433000


In [24]:
ratings_2026 = ratings_2026_p1.copy()

# Remove the non-game/header row
ratings_2026 = ratings_2026.dropna(subset=["figure"]).copy()

# Convert date to datetime
ratings_2026["date"] = pd.to_datetime(
    ratings_2026["date"],
    errors="coerce"
)

# Remove anything that isn't a valid date
ratings_2026 = ratings_2026.dropna(subset=["date"]).copy()

# Convert audience figure to integer
ratings_2026["figure"] = ratings_2026["figure"].astype(int)

# Sort chronologically
ratings_2026 = ratings_2026.sort_values("date").reset_index(drop=True)

print("Rows:", len(ratings_2026))
print("Date range:", ratings_2026["date"].min(), "to", ratings_2026["date"].max())

Rows: 59
Date range: 2026-01-24 00:00:00 to 2026-06-13 00:00:00


In [25]:
ratings_2025_26 = pd.concat(
    [ratings_2025, ratings_2026],
    ignore_index=True
)

ratings_2025_26 = (
    ratings_2025_26
    .sort_values("date")
    .reset_index(drop=True)
)

print("Total ratings records:", len(ratings_2025_26))
print(
    "Date range:",
    ratings_2025_26["date"].min(),
    "to",
    ratings_2025_26["date"].max()
)

display(ratings_2025_26)

Total ratings records: 74
Date range: 2025-10-21 00:00:00 to 2026-06-13 00:00:00


,date,telecast,figure
0,2025-10-21,NBA Rockets–Thunder,5900000
1,2025-10-22,NBA Cavaliers–Knicks,2170000
2,2025-10-22,NBA Spurs–Mavericks,2490000
3,2025-10-29,NBA Cavaliers–Celtics,1280000
4,2025-10-29,NBA Lakers–Timberwolves,1360000
5,2025-10-31,NBA Celtics–76ers,961000
6,2025-10-31,NBA Lakers–Grizzlies,1200000
7,2025-11-14,NBA Warriors–Spurs,1430000
8,2025-11-26,Minnesota Timberwolves at Oklahoma City Thunder,2433000
9,2025-11-26,Houston Rockets at Golden State Warriors,2031000


In [27]:
import pandas as pd

nba_schedule = pd.read_csv(
    "NBA_Global_Media_Insights/Data/raw/nba_2025_26_schedule.csv"
)

print("Shape:", nba_schedule.shape)
print("\nColumns:")
print(nba_schedule.columns.tolist())
display(nba_schedule.head(10))

Shape: (1278, 14)

Columns:
['game_id', 'season_type', 'tip_utc', 'tip_et', 'away_team', 'home_team', 'venue', 'city', 'state', 'national_tv', 'home_rsns', 'away_rsns', 'international', 'is_tbd']


,game_id,season_type,tip_utc,tip_et,away_team,home_team,venue,city,state,national_tv,home_rsns,away_rsns,international,is_tbd
0,12500008,Preseason,2025-10-02T16:00:00Z,2025-10-02T11:00:00-05:00,PHI (76ers),NYK (Knicks),Etihad Arena,Abu Dhabi,NaN,NBA TV,MSG,NaN,NaN,False
1,12500001,Preseason,2025-10-04T02:00:00Z,2025-10-03T21:00:00-05:00,PHX (Suns),LAL (Lakers),Acrisure Arena,Palm Desert,CA,NBA TV,Spectrum SportsNet,Arizona's Family 3TV / Arizona's Family Sports,NaN,False
2,12500009,Preseason,2025-10-03T09:30:00Z,2025-10-03T04:30:00-05:00,MEL (United),NOP (Pelicans),Rod Laver Arena,Melbourne,NaN,NBA TV,GCSEN,NaN,NaN,False
3,12500010,Preseason,2025-10-04T15:00:00Z,2025-10-04T10:00:00-05:00,NYK (Knicks),PHI (76ers),Etihad Arena,Abu Dhabi,NaN,NBA TV,NaN,MSG,NaN,False
4,12500011,Preseason,2025-10-05T03:00:00Z,2025-10-04T22:00:00-05:00,SEM (Phoenix),NOP (Pelicans),Rod Laver Arena,Melbourne,NaN,NBA TV,GCSEN,NaN,NaN,False
5,12500026,Preseason,2025-10-05T00:00:00Z,2025-10-04T19:00:00-05:00,HAP (Jerusalem B.C.),BKN (Nets),Barclays Center,Brooklyn,NY,NaN,YES,NaN,NaN,False
6,12500027,Preseason,2025-10-05T00:00:00Z,2025-10-04T19:00:00-05:00,ORL (Magic),MIA (Heat),Coliseo de Puerto Rico,"San Juan,Puerto Rico",NaN,NBA TV,FanDuel Sports Network - Sun,FanDuel Sports Network - Florida,NaN,False
7,12500028,Preseason,2025-10-05T01:00:00Z,2025-10-04T20:00:00-05:00,MIN (Timberwolves),DEN (Nuggets),Pechanga Arena,San Diego,CA,NaN,Altitude,NaN,NaN,False
8,12500029,Preseason,2025-10-05T21:00:00Z,2025-10-05T16:00:00-05:00,OKC (Thunder),CHA (Hornets),North Charleston Coliseum,Charleston,SC,NaN,NaN,Thunder App,NaN,False
9,12500030,Preseason,2025-10-06T00:30:00Z,2025-10-05T19:30:00-05:00,LAL (Lakers),GSW (Warriors),Chase Center,San Francisco,CA,ESPN,NBCSBA,Spectrum SportsNet,NaN,False


In [28]:
# Make a copy so we preserve the original schedule
schedule_match = nba_schedule.copy()

# Extract the Eastern Time calendar date
schedule_match["date"] = pd.to_datetime(
    schedule_match["tip_et"],
    errors="coerce"
).dt.date

# Extract the 3-letter team abbreviation from values like:
# "PHI (76ers)" -> "PHI"
# "NYK (Knicks)" -> "NYK"
schedule_match["away_abbr"] = (
    schedule_match["away_team"]
    .str.extract(r"^([A-Z]{3})", expand=False)
)

schedule_match["home_abbr"] = (
    schedule_match["home_team"]
    .str.extract(r"^([A-Z]{3})", expand=False)
)

# Create a matching key
schedule_match["match_key"] = (
    schedule_match["date"].astype(str)
    + "_"
    + schedule_match["away_abbr"]
    + "_"
    + schedule_match["home_abbr"]
)

print("Schedule rows:", len(schedule_match))

display(
    schedule_match[
        ["date", "away_team", "home_team", "away_abbr", "home_abbr",
         "national_tv", "match_key"]
    ].head(10)
)

Schedule rows: 1278


,date,away_team,home_team,away_abbr,home_abbr,national_tv,match_key
0,2025-10-02,PHI (76ers),NYK (Knicks),PHI,NYK,NBA TV,2025-10-02_PHI_NYK
1,2025-10-03,PHX (Suns),LAL (Lakers),PHX,LAL,NBA TV,2025-10-03_PHX_LAL
2,2025-10-03,MEL (United),NOP (Pelicans),MEL,NOP,NBA TV,2025-10-03_MEL_NOP
3,2025-10-04,NYK (Knicks),PHI (76ers),NYK,PHI,NBA TV,2025-10-04_NYK_PHI
4,2025-10-04,SEM (Phoenix),NOP (Pelicans),SEM,NOP,NBA TV,2025-10-04_SEM_NOP
5,2025-10-04,HAP (Jerusalem B.C.),BKN (Nets),HAP,BKN,NaN,2025-10-04_HAP_BKN
6,2025-10-04,ORL (Magic),MIA (Heat),ORL,MIA,NBA TV,2025-10-04_ORL_MIA
7,2025-10-04,MIN (Timberwolves),DEN (Nuggets),MIN,DEN,NaN,2025-10-04_MIN_DEN
8,2025-10-05,OKC (Thunder),CHA (Hornets),OKC,CHA,NaN,2025-10-05_OKC_CHA
9,2025-10-05,LAL (Lakers),GSW (Warriors),LAL,GSW,ESPN,2025-10-05_LAL_GSW


In [29]:
team_map = {
    "Hawks": "ATL",
    "Celtics": "BOS",
    "Nets": "BKN",
    "Hornets": "CHA",
    "Bulls": "CHI",
    "Cavaliers": "CLE",
    "Mavericks": "DAL",
    "Nuggets": "DEN",
    "Pistons": "DET",
    "Warriors": "GSW",
    "Rockets": "HOU",
    "Pacers": "IND",
    "Clippers": "LAC",
    "Lakers": "LAL",
    "Grizzlies": "MEM",
    "Heat": "MIA",
    "Bucks": "MIL",
    "Timberwolves": "MIN",
    "Pelicans": "NOP",
    "Knicks": "NYK",
    "Thunder": "OKC",
    "Magic": "ORL",
    "76ers": "PHI",
    "Suns": "PHX",
    "Blazers": "POR",
    "Kings": "SAC",
    "Spurs": "SAS",
    "Raptors": "TOR",
    "Jazz": "UTA",
    "Wizards": "WAS",
}

print("Teams in dictionary:", len(team_map))

Teams in dictionary: 30


In [30]:
import re

def extract_teams(telecast):
    found = []

    for team_name, abbr in team_map.items():
        if re.search(rf"\b{re.escape(team_name)}\b", telecast, re.IGNORECASE):
            found.append(abbr)

    return found

ratings_2025_26["teams_found"] = ratings_2025_26["telecast"].apply(extract_teams)

display(
    ratings_2025_26[
        ["date", "telecast", "teams_found", "figure"]
    ]
)

,date,telecast,teams_found,figure
0,2025-10-21,NBA Rockets–Thunder,"[HOU, OKC]",5900000
1,2025-10-22,NBA Cavaliers–Knicks,"[CLE, NYK]",2170000
2,2025-10-22,NBA Spurs–Mavericks,"[DAL, SAS]",2490000
3,2025-10-29,NBA Cavaliers–Celtics,"[BOS, CLE]",1280000
4,2025-10-29,NBA Lakers–Timberwolves,"[LAL, MIN]",1360000
5,2025-10-31,NBA Celtics–76ers,"[BOS, PHI]",961000
6,2025-10-31,NBA Lakers–Grizzlies,"[LAL, MEM]",1200000
7,2025-11-14,NBA Warriors–Spurs,"[GSW, SAS]",1430000
8,2025-11-26,Minnesota Timberwolves at Oklahoma City Thunder,"[MIN, OKC]",2433000
9,2025-11-26,Houston Rockets at Golden State Warriors,"[GSW, HOU]",2031000


In [31]:
# Create a normalized key based on date + the two teams,
# regardless of which team is listed first.

ratings_2025_26["team_key"] = ratings_2025_26["teams_found"].apply(
    lambda x: "_".join(sorted(x)) if len(x) == 2 else None
)

ratings_2025_26["match_key"] = (
    ratings_2025_26["date"].astype(str)
    + "_"
    + ratings_2025_26["team_key"].fillna("")
)

display(
    ratings_2025_26[
        ["date", "telecast", "teams_found", "team_key", "match_key", "figure"]
    ]
)

,date,telecast,teams_found,team_key,match_key,figure
0,2025-10-21,NBA Rockets–Thunder,"[HOU, OKC]",HOU_OKC,2025-10-21_HOU_OKC,5900000
1,2025-10-22,NBA Cavaliers–Knicks,"[CLE, NYK]",CLE_NYK,2025-10-22_CLE_NYK,2170000
2,2025-10-22,NBA Spurs–Mavericks,"[DAL, SAS]",DAL_SAS,2025-10-22_DAL_SAS,2490000
3,2025-10-29,NBA Cavaliers–Celtics,"[BOS, CLE]",BOS_CLE,2025-10-29_BOS_CLE,1280000
4,2025-10-29,NBA Lakers–Timberwolves,"[LAL, MIN]",LAL_MIN,2025-10-29_LAL_MIN,1360000
5,2025-10-31,NBA Celtics–76ers,"[BOS, PHI]",BOS_PHI,2025-10-31_BOS_PHI,961000
6,2025-10-31,NBA Lakers–Grizzlies,"[LAL, MEM]",LAL_MEM,2025-10-31_LAL_MEM,1200000
7,2025-11-14,NBA Warriors–Spurs,"[GSW, SAS]",GSW_SAS,2025-11-14_GSW_SAS,1430000
8,2025-11-26,Minnesota Timberwolves at Oklahoma City Thunder,"[MIN, OKC]",MIN_OKC,2025-11-26_MIN_OKC,2433000
9,2025-11-26,Houston Rockets at Golden State Warriors,"[GSW, HOU]",GSW_HOU,2025-11-26_GSW_HOU,2031000


In [33]:
schedule_match["team_key"] = (
    schedule_match[["away_abbr", "home_abbr"]]
    .apply(
        lambda row: "_".join(
            sorted([x for x in row if pd.notna(x)])
        ),
        axis=1
    )
)

schedule_match["match_key"] = (
    schedule_match["date"].astype(str)
    + "_"
    + schedule_match["team_key"]
)

display(
    schedule_match[
        ["date", "away_team", "home_team", "away_abbr", "home_abbr",
         "team_key", "match_key", "national_tv"]
    ].head(10)
)

,date,away_team,home_team,away_abbr,home_abbr,team_key,match_key,national_tv
0,2025-10-02,PHI (76ers),NYK (Knicks),PHI,NYK,NYK_PHI,2025-10-02_NYK_PHI,NBA TV
1,2025-10-03,PHX (Suns),LAL (Lakers),PHX,LAL,LAL_PHX,2025-10-03_LAL_PHX,NBA TV
2,2025-10-03,MEL (United),NOP (Pelicans),MEL,NOP,MEL_NOP,2025-10-03_MEL_NOP,NBA TV
3,2025-10-04,NYK (Knicks),PHI (76ers),NYK,PHI,NYK_PHI,2025-10-04_NYK_PHI,NBA TV
4,2025-10-04,SEM (Phoenix),NOP (Pelicans),SEM,NOP,NOP_SEM,2025-10-04_NOP_SEM,NBA TV
5,2025-10-04,HAP (Jerusalem B.C.),BKN (Nets),HAP,BKN,BKN_HAP,2025-10-04_BKN_HAP,NaN
6,2025-10-04,ORL (Magic),MIA (Heat),ORL,MIA,MIA_ORL,2025-10-04_MIA_ORL,NBA TV
7,2025-10-04,MIN (Timberwolves),DEN (Nuggets),MIN,DEN,DEN_MIN,2025-10-04_DEN_MIN,NaN
8,2025-10-05,OKC (Thunder),CHA (Hornets),OKC,CHA,CHA_OKC,2025-10-05_CHA_OKC,NaN
9,2025-10-05,LAL (Lakers),GSW (Warriors),LAL,GSW,GSW_LAL,2025-10-05_GSW_LAL,ESPN


In [34]:
print("Rows with missing team abbreviation:")

display(
    schedule_match[
        schedule_match["away_abbr"].isna() |
        schedule_match["home_abbr"].isna()
    ][
        ["date", "away_team", "home_team", "away_abbr", "home_abbr"]
    ]
)

Rows with missing team abbreviation:


,date,away_team,home_team,away_abbr,home_abbr
428,2025-12-09,(),(),NaN,NaN
429,2025-12-09,(),(),NaN,NaN
430,2025-12-10,(),(),NaN,NaN
431,2025-12-10,(),(),NaN,NaN
432,2025-12-13,(),(),NaN,NaN
433,2025-12-13,(),(),NaN,NaN
434,2025-12-16,(),(),NaN,NaN


In [35]:
affected_dates = schedule_match[
    schedule_match["away_abbr"].isna() |
    schedule_match["home_abbr"].isna()
]["date"].unique()

display(
    schedule_match[
        schedule_match["date"].isin(affected_dates)
    ][
        ["date", "game_id", "away_team", "home_team",
         "away_abbr", "home_abbr", "national_tv"]
    ].sort_values(["date", "game_id"])
)

,date,game_id,away_team,home_team,away_abbr,home_abbr,national_tv
428,2025-12-09,22501201,(),(),NaN,NaN,Prime Video
429,2025-12-09,22501203,(),(),NaN,NaN,Prime Video
430,2025-12-10,22501202,(),(),NaN,NaN,Prime Video
431,2025-12-10,22501204,(),(),NaN,NaN,Prime Video
432,2025-12-13,22501229,(),(),NaN,NaN,Prime Video
433,2025-12-13,22501230,(),(),NaN,NaN,Prime Video
434,2025-12-16,62500001,(),(),NaN,NaN,Prime Video


In [36]:
schedule_match = schedule_match.dropna(
    subset=["away_abbr", "home_abbr"]
).copy()

print("Schedule rows remaining:", len(schedule_match))
print("Rows with missing team abbreviations:",
      schedule_match["away_abbr"].isna().sum() +
      schedule_match["home_abbr"].isna().sum())

Schedule rows remaining: 1271
Rows with missing team abbreviations: 0


In [37]:
duplicate_keys = schedule_match[
    schedule_match["match_key"].duplicated(keep=False)
].sort_values("match_key")

print("Duplicate match keys:", duplicate_keys["match_key"].nunique())

display(
    duplicate_keys[
        ["date", "game_id", "away_team", "home_team",
         "team_key", "match_key", "national_tv"]
    ]
)

Duplicate match keys: 0


,date,game_id,away_team,home_team,team_key,match_key,national_tv


In [38]:
ratings_matched = ratings_2025_26.merge(
    schedule_match[
        [
            "match_key",
            "game_id",
            "season_type",
            "tip_et",
            "away_team",
            "home_team",
            "national_tv"
        ]
    ],
    on="match_key",
    how="left",
    indicator=True
)

print("Total ratings records:", len(ratings_matched))
print(
    "Matched:",
    (ratings_matched["_merge"] == "both").sum()
)
print(
    "Unmatched:",
    (ratings_matched["_merge"] == "left_only").sum()
)

display(
    ratings_matched[
        [
            "date",
            "telecast",
            "figure",
            "away_team",
            "home_team",
            "national_tv",
            "_merge"
        ]
    ]
)

Total ratings records: 74
Matched: 36
Unmatched: 38


,date,telecast,figure,away_team,home_team,national_tv,_merge
0,2025-10-21,NBA Rockets–Thunder,5900000,HOU (Rockets),OKC (Thunder),NBC | Peacock,both
1,2025-10-22,NBA Cavaliers–Knicks,2170000,CLE (Cavaliers),NYK (Knicks),ESPN,both
2,2025-10-22,NBA Spurs–Mavericks,2490000,SAS (Spurs),DAL (Mavericks),ESPN,both
3,2025-10-29,NBA Cavaliers–Celtics,1280000,CLE (Cavaliers),BOS (Celtics),ESPN,both
4,2025-10-29,NBA Lakers–Timberwolves,1360000,LAL (Lakers),MIN (Timberwolves),ESPN,both
5,2025-10-31,NBA Celtics–76ers,961000,BOS (Celtics),PHI (76ers),Prime Video,both
6,2025-10-31,NBA Lakers–Grizzlies,1200000,LAL (Lakers),MEM (Grizzlies),Prime Video,both
7,2025-11-14,NBA Warriors–Spurs,1430000,GSW (Warriors),SAS (Spurs),Prime Video,both
8,2025-11-26,Minnesota Timberwolves at Oklahoma City Thunder,2433000,MIN (Timberwolves),OKC (Thunder),ESPN,both
9,2025-11-26,Houston Rockets at Golden State Warriors,2031000,HOU (Rockets),GSW (Warriors),ESPN,both


In [39]:
playoff_schedule = nba_schedule[
    pd.to_datetime(nba_schedule["tip_et"], errors="coerce").dt.date
    >= pd.to_datetime("2026-04-14").date()
].copy()

print("Playoff-period schedule rows:", len(playoff_schedule))

display(
    playoff_schedule[
        [
            "game_id",
            "season_type",
            "tip_et",
            "away_team",
            "home_team",
            "national_tv"
        ]
    ].head(30)
)

Playoff-period schedule rows: 0


,game_id,season_type,tip_et,away_team,home_team,national_tv


In [40]:
print("Rows with missing away team:",
      playoff_schedule["away_team"].isna().sum())

print("Rows with missing home team:",
      playoff_schedule["home_team"].isna().sum())

print("Unique season types:")
print(playoff_schedule["season_type"].value_counts(dropna=False))

Rows with missing away team: 0
Rows with missing home team: 0
Unique season types:
Series([], Name: season_type, dtype: int64)


In [41]:
print("Schedule date range:")
print("Earliest:", pd.to_datetime(nba_schedule["tip_et"], errors="coerce").min())
print("Latest:  ", pd.to_datetime(nba_schedule["tip_et"], errors="coerce").max())

Schedule date range:
Earliest: 2025-10-02 11:00:00-05:00
Latest:   2026-04-12 19:30:00-05:00


In [42]:
display(
    nba_schedule[
        pd.to_datetime(nba_schedule["tip_et"], errors="coerce")
        == pd.to_datetime(nba_schedule["tip_et"], errors="coerce").max()
    ][
        ["game_id", "season_type", "tip_et",
         "away_team", "home_team", "national_tv"]
    ]
)

,game_id,season_type,tip_et,away_team,home_team,national_tv
1270,22501193,Regular Season,2026-04-12T19:30:00-05:00,CHI (Bulls),DAL (Mavericks),NaN
1271,22501194,Regular Season,2026-04-12T19:30:00-05:00,MEM (Grizzlies),HOU (Rockets),NaN
1272,22501195,Regular Season,2026-04-12T19:30:00-05:00,NOP (Pelicans),MIN (Timberwolves),NaN
1273,22501196,Regular Season,2026-04-12T19:30:00-05:00,PHX (Suns),OKC (Thunder),NaN
1274,22501197,Regular Season,2026-04-12T19:30:00-05:00,DEN (Nuggets),SAS (Spurs),NaN
1275,22501198,Regular Season,2026-04-12T19:30:00-05:00,UTA (Jazz),LAL (Lakers),NaN
1276,22501199,Regular Season,2026-04-12T19:30:00-05:00,GSW (Warriors),LAC (Clippers),NaN
1277,22501200,Regular Season,2026-04-12T19:30:00-05:00,SAC (Kings),POR (Trail Blazers),NaN


In [43]:
ratings_regular = ratings_matched[
    ratings_matched["_merge"] == "both"
].copy()

print("Matched regular-season ratings:", len(ratings_regular))

display(
    ratings_regular[
        [
            "date",
            "telecast",
            "figure",
            "game_id",
            "away_team",
            "home_team",
            "national_tv"
        ]
    ]
)

Matched regular-season ratings: 36


,date,telecast,figure,game_id,away_team,home_team,national_tv
0,2025-10-21,NBA Rockets–Thunder,5900000,22500001.0,HOU (Rockets),OKC (Thunder),NBC | Peacock
1,2025-10-22,NBA Cavaliers–Knicks,2170000,22500003.0,CLE (Cavaliers),NYK (Knicks),ESPN
2,2025-10-22,NBA Spurs–Mavericks,2490000,22500004.0,SAS (Spurs),DAL (Mavericks),ESPN
3,2025-10-29,NBA Cavaliers–Celtics,1280000,22500128.0,CLE (Cavaliers),BOS (Celtics),ESPN
4,2025-10-29,NBA Lakers–Timberwolves,1360000,22500136.0,LAL (Lakers),MIN (Timberwolves),ESPN
5,2025-10-31,NBA Celtics–76ers,961000,22500021.0,BOS (Celtics),PHI (76ers),Prime Video
6,2025-10-31,NBA Lakers–Grizzlies,1200000,22500024.0,LAL (Lakers),MEM (Grizzlies),Prime Video
7,2025-11-14,NBA Warriors–Spurs,1430000,22500047.0,GSW (Warriors),SAS (Spurs),Prime Video
8,2025-11-26,Minnesota Timberwolves at Oklahoma City Thunder,2433000,22500064.0,MIN (Timberwolves),OKC (Thunder),ESPN
9,2025-11-26,Houston Rockets at Golden State Warriors,2031000,22500066.0,HOU (Rockets),GSW (Warriors),ESPN


In [44]:
ratings_unmatched = ratings_matched[
    ratings_matched["_merge"] == "left_only"
].copy()

print("Unmatched ratings:", len(ratings_unmatched))

display(
    ratings_unmatched[
        ["date", "telecast", "figure"]
    ]
)

Unmatched ratings: 38


,date,telecast,figure
10,2025-12-09,New York Knicks at Toronto Raptors,1140000
11,2025-12-16,San Antonio Spurs at New York Knicks,3070000
38,2026-04-14,Miami Heat at Charlotte Hornets,3070000
39,2026-04-18,Houston Rockets at Los Angeles Lakers,5200000
40,2026-04-19,Portland Trail Blazers at San Antonio Spurs,4970000
41,2026-04-20,Minnesota Timberwolves at Denver Nuggets,3300000
42,2026-04-20,Atlanta Hawks at New York Knicks,3880000
43,2026-04-21,Portland Trail Blazers at San Antonio Spurs,3840000
44,2026-04-21,Houston Rockets at Los Angeles Lakers,3760000
45,2026-04-25,Oklahoma City Thunder at Phoenix Suns,3300000


In [47]:
playoff_url = "https://www.basketball-reference.com/playoffs/NBA_2026_games.html"

playoff_tables = pd.read_html(playoff_url)

print("Number of tables:", len(playoff_tables))

display(playoff_tables[0].head())

Number of tables: 1


,Date,Start (ET),Visitor/Neutral,PTS,Home/Neutral,PTS.1,Unnamed: 6,Unnamed: 7,Attend.,LOG,Arena,Notes
0,"Sat, Apr 18, 2026",1:00p,Toronto Raptors,113,Cleveland Cavaliers,126,Box Score,NaN,19432,2:36,Rocket Arena,NaN
1,"Sat, Apr 18, 2026",3:30p,Minnesota Timberwolves,105,Denver Nuggets,116,Box Score,NaN,19796,2:33,Ball Arena,NaN
2,"Sat, Apr 18, 2026",6:00p,Atlanta Hawks,102,New York Knicks,113,Box Score,NaN,19812,2:32,Madison Square Garden (IV),NaN
3,"Sat, Apr 18, 2026",8:30p,Houston Rockets,98,Los Angeles Lakers,107,Box Score,NaN,19057,2:31,Crypto.com Arena,NaN
4,"Sun, Apr 19, 2026",1:00p,Philadelphia 76ers,91,Boston Celtics,123,Box Score,NaN,19156,2:24,TD Garden,NaN


In [48]:
# Copy the Basketball-Reference table
playoff_schedule = playoff_tables[0].copy()

# Keep only the columns we need
playoff_schedule = playoff_schedule[
    ["Date", "Start (ET)", "Visitor/Neutral", "Home/Neutral"]
].copy()

# Convert date to the same format used by ratings_2025_26
playoff_schedule["date"] = pd.to_datetime(
    playoff_schedule["Date"],
    errors="coerce"
).dt.date

# Map full team names to NBA abbreviations
team_map_reverse = {
    "Atlanta Hawks": "ATL",
    "Boston Celtics": "BOS",
    "Brooklyn Nets": "BKN",
    "Charlotte Hornets": "CHA",
    "Chicago Bulls": "CHI",
    "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL",
    "Denver Nuggets": "DEN",
    "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW",
    "Houston Rockets": "HOU",
    "Indiana Pacers": "IND",
    "Los Angeles Clippers": "LAC",
    "Los Angeles Lakers": "LAL",
    "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA",
    "Milwaukee Bucks": "MIL",
    "Minnesota Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP",
    "New York Knicks": "NYK",
    "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL",
    "Philadelphia 76ers": "PHI",
    "Phoenix Suns": "PHX",
    "Portland Trail Blazers": "POR",
    "Sacramento Kings": "SAC",
    "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR",
    "Utah Jazz": "UTA",
    "Washington Wizards": "WAS"
}

playoff_schedule["away_abbr"] = playoff_schedule["Visitor/Neutral"].map(
    team_map_reverse
)

playoff_schedule["home_abbr"] = playoff_schedule["Home/Neutral"].map(
    team_map_reverse
)

# Create the same order-independent team key
playoff_schedule["team_key"] = (
    playoff_schedule[["away_abbr", "home_abbr"]]
    .apply(
        lambda row: "_".join(
            sorted([x for x in row if pd.notna(x)])
        ),
        axis=1
    )
)

# Create the same match key used by the ratings data
playoff_schedule["match_key"] = (
    playoff_schedule["date"].astype(str)
    + "_"
    + playoff_schedule["team_key"]
)

print("Playoff games:", len(playoff_schedule))
print(
    "Missing team abbreviations:",
    playoff_schedule["away_abbr"].isna().sum()
    + playoff_schedule["home_abbr"].isna().sum()
)

display(
    playoff_schedule[
        [
            "date",
            "Visitor/Neutral",
            "Home/Neutral",
            "away_abbr",
            "home_abbr",
            "team_key",
            "match_key"
        ]
    ].head(10)
)

Playoff games: 85
Missing team abbreviations: 0


,date,Visitor/Neutral,Home/Neutral,away_abbr,home_abbr,team_key,match_key
0,2026-04-18,Toronto Raptors,Cleveland Cavaliers,TOR,CLE,CLE_TOR,2026-04-18_CLE_TOR
1,2026-04-18,Minnesota Timberwolves,Denver Nuggets,MIN,DEN,DEN_MIN,2026-04-18_DEN_MIN
2,2026-04-18,Atlanta Hawks,New York Knicks,ATL,NYK,ATL_NYK,2026-04-18_ATL_NYK
3,2026-04-18,Houston Rockets,Los Angeles Lakers,HOU,LAL,HOU_LAL,2026-04-18_HOU_LAL
4,2026-04-19,Philadelphia 76ers,Boston Celtics,PHI,BOS,BOS_PHI,2026-04-19_BOS_PHI
5,2026-04-19,Phoenix Suns,Oklahoma City Thunder,PHX,OKC,OKC_PHX,2026-04-19_OKC_PHX
6,2026-04-19,Orlando Magic,Detroit Pistons,ORL,DET,DET_ORL,2026-04-19_DET_ORL
7,2026-04-19,Portland Trail Blazers,San Antonio Spurs,POR,SAS,POR_SAS,2026-04-19_POR_SAS
8,2026-04-20,Toronto Raptors,Cleveland Cavaliers,TOR,CLE,CLE_TOR,2026-04-20_CLE_TOR
9,2026-04-20,Atlanta Hawks,New York Knicks,ATL,NYK,ATL_NYK,2026-04-20_ATL_NYK


In [49]:
# Keep the columns we need from the existing regular-season schedule
regular_schedule = schedule_match[
    [
        "date",
        "match_key",
        "game_id",
        "season_type",
        "tip_et",
        "away_team",
        "home_team",
        "national_tv"
    ]
].copy()

# Rename playoff columns to match
playoff_schedule_clean = playoff_schedule[
    [
        "date",
        "match_key",
        "Visitor/Neutral",
        "Home/Neutral"
    ]
].copy()

playoff_schedule_clean = playoff_schedule_clean.rename(
    columns={
        "Visitor/Neutral": "away_team",
        "Home/Neutral": "home_team"
    }
)

# Add fields that don't exist in the Basketball-Reference table
playoff_schedule_clean["game_id"] = pd.NA
playoff_schedule_clean["season_type"] = "Playoffs"
playoff_schedule_clean["tip_et"] = pd.NA
playoff_schedule_clean["national_tv"] = pd.NA

# Put columns in the same order
playoff_schedule_clean = playoff_schedule_clean[
    [
        "date",
        "match_key",
        "game_id",
        "season_type",
        "tip_et",
        "away_team",
        "home_team",
        "national_tv"
    ]
]

# Combine the two schedules
master_schedule = pd.concat(
    [regular_schedule, playoff_schedule_clean],
    ignore_index=True
)

print("Master schedule rows:", len(master_schedule))
print("Regular season rows:", len(regular_schedule))
print("Playoff rows:", len(playoff_schedule_clean))

print("\nDuplicate match keys:")
print(master_schedule["match_key"].duplicated().sum())

Master schedule rows: 1356
Regular season rows: 1271
Playoff rows: 85

Duplicate match keys:
0


In [50]:
ratings_rematched = ratings_2025_26.merge(
    master_schedule[
        [
            "match_key",
            "game_id",
            "season_type",
            "tip_et",
            "away_team",
            "home_team",
            "national_tv"
        ]
    ],
    on="match_key",
    how="left",
    indicator=True
)

print("Total ratings:", len(ratings_rematched))
print("Matched:", (ratings_rematched["_merge"] == "both").sum())
print("Unmatched:", (ratings_rematched["_merge"] == "left_only").sum())

display(
    ratings_rematched[
        ratings_rematched["_merge"] == "left_only"
    ][
        ["date", "telecast", "figure"]
    ]
)

Total ratings: 74
Matched: 64
Unmatched: 10


,date,telecast,figure
10,2025-12-09,New York Knicks at Toronto Raptors,1140000
11,2025-12-16,San Antonio Spurs at New York Knicks,3070000
38,2026-04-14,Miami Heat at Charlotte Hornets,3070000
55,2026-05-04,NBA Knicks–Sixers,6300000
65,2026-05-20,Western Conference Finals Game 2,7770000
69,2026-06-03,NBA Finals Game 1,16930000
70,2026-06-05,NBA Finals Game 2,16430000
71,2026-06-08,NBA Finals Game 3,23790000
72,2026-06-10,NBA Finals Game 4,20940000
73,2026-06-13,NBA Finals Game 5,24540000


In [51]:
# Manually add team keys for the remaining identifiable telecasts
ratings_rematched.loc[
    (ratings_rematched["date"] == pd.to_datetime("2025-12-09").date()),
    "match_key"
] = "2025-12-09_NYK_TOR"

ratings_rematched.loc[
    (ratings_rematched["date"] == pd.to_datetime("2025-12-16").date()),
    "match_key"
] = "2025-12-16_NYK_SAS"

ratings_rematched.loc[
    (ratings_rematched["date"] == pd.to_datetime("2026-04-14").date()),
    "match_key"
] = "2026-04-14_CHA_MIA"

ratings_rematched.loc[
    (ratings_rematched["date"] == pd.to_datetime("2026-05-04").date()),
    "match_key"
] = "2026-05-04_NYK_PHI"

In [52]:
# Build a date -> match_key lookup from the playoff schedule
playoff_date_lookup = (
    playoff_schedule
    .groupby("date")["match_key"]
    .agg(list)
    .to_dict()
)

generic_dates = [
    pd.to_datetime("2026-05-20").date(),
    pd.to_datetime("2026-06-03").date(),
    pd.to_datetime("2026-06-05").date(),
    pd.to_datetime("2026-06-08").date(),
    pd.to_datetime("2026-06-10").date(),
    pd.to_datetime("2026-06-13").date()
]

for d in generic_dates:
    keys = playoff_date_lookup.get(d, [])
    
    if len(keys) == 1:
        ratings_rematched.loc[
            ratings_rematched["date"] == d,
            "match_key"
        ] = keys[0]
    else:
        print(f"{d}: found {len(keys)} playoff games -> {keys}")

In [53]:
# Rebuild corrected match keys on the ratings data
ratings_corrected = ratings_2025_26.copy()

# Fix identifiable exceptions
ratings_corrected.loc[
    (ratings_corrected["date"] == pd.to_datetime("2025-12-09").date()),
    "match_key"
] = "2025-12-09_NYK_TOR"

ratings_corrected.loc[
    (ratings_corrected["date"] == pd.to_datetime("2025-12-16").date()),
    "match_key"
] = "2025-12-16_NYK_SAS"

ratings_corrected.loc[
    (ratings_corrected["date"] == pd.to_datetime("2026-04-14").date()),
    "match_key"
] = "2026-04-14_CHA_MIA"

ratings_corrected.loc[
    (ratings_corrected["date"] == pd.to_datetime("2026-05-04").date()),
    "match_key"
] = "2026-05-04_NYK_PHI"

# Fix generic playoff games by date
for d in generic_dates:
    keys = playoff_date_lookup.get(d, [])
    if len(keys) == 1:
        ratings_corrected.loc[
            ratings_corrected["date"] == d,
            "match_key"
        ] = keys[0]

# Merge with master schedule
ratings_final = ratings_corrected.merge(
    master_schedule[
        [
            "match_key",
            "game_id",
            "season_type",
            "tip_et",
            "away_team",
            "home_team",
            "national_tv"
        ]
    ],
    on="match_key",
    how="left",
    indicator=True
)

print("Total ratings:", len(ratings_final))
print("Matched:", (ratings_final["_merge"] == "both").sum())
print("Unmatched:", (ratings_final["_merge"] == "left_only").sum())

display(
    ratings_final[
        ratings_final["_merge"] == "left_only"
    ][
        ["date", "telecast", "figure", "match_key"]
    ]
)

Total ratings: 74
Matched: 64
Unmatched: 10


,date,telecast,figure,match_key
10,2025-12-09,New York Knicks at Toronto Raptors,1140000,2025-12-09_NYK_TOR
11,2025-12-16,San Antonio Spurs at New York Knicks,3070000,2025-12-16_NYK_SAS
38,2026-04-14,Miami Heat at Charlotte Hornets,3070000,2026-04-14_CHA_MIA
55,2026-05-04,NBA Knicks–Sixers,6300000,2026-05-04_
65,2026-05-20,Western Conference Finals Game 2,7770000,2026-05-20_
69,2026-06-03,NBA Finals Game 1,16930000,2026-06-03_
70,2026-06-05,NBA Finals Game 2,16430000,2026-06-05_
71,2026-06-08,NBA Finals Game 3,23790000,2026-06-08_
72,2026-06-10,NBA Finals Game 4,20940000,2026-06-10_
73,2026-06-13,NBA Finals Game 5,24540000,2026-06-13_


In [58]:
# Start fresh from the original ratings data
ratings_corrected = ratings_2025_26.copy()

# Manually correct the remaining identifiable games
manual_keys = {
    pd.to_datetime("2025-12-09").date(): "2025-12-09_NYK_TOR",
    pd.to_datetime("2025-12-16").date(): "2025-12-16_NYK_SAS",
    pd.to_datetime("2026-04-14").date(): "2026-04-14_CHA_MIA",
    pd.to_datetime("2026-05-04").date(): "2026-05-04_NYK_PHI",
    pd.to_datetime("2026-05-20").date(): "2026-05-20_OKC_SAS",
    pd.to_datetime("2026-06-03").date(): "2026-06-03_NYK_SAS",
    pd.to_datetime("2026-06-05").date(): "2026-06-05_NYK_SAS",
    pd.to_datetime("2026-06-08").date(): "2026-06-08_NYK_SAS",
    pd.to_datetime("2026-06-10").date(): "2026-06-10_NYK_SAS",
    pd.to_datetime("2026-06-13").date(): "2026-06-13_NYK_SAS"
}

for date, key in manual_keys.items():
    ratings_corrected.loc[
        ratings_corrected["date"] == date,
        "match_key"
    ] = key

# Merge with the master schedule
ratings_final = ratings_corrected.merge(
    master_schedule[
        [
            "match_key",
            "game_id",
            "season_type",
            "tip_et",
            "away_team",
            "home_team",
            "national_tv"
        ]
    ],
    on="match_key",
    how="left",
    indicator=True
)

print("Total ratings:", len(ratings_final))
print("Matched:", (ratings_final["_merge"] == "both").sum())
print("Unmatched:", (ratings_final["_merge"] == "left_only").sum())

display(
    ratings_final[
        ratings_final["_merge"] == "left_only"
    ][
        ["date", "telecast", "figure", "match_key"]
    ]
)

Total ratings: 74
Matched: 64
Unmatched: 10


,date,telecast,figure,match_key
10,2025-12-09,New York Knicks at Toronto Raptors,1140000,2025-12-09_NYK_TOR
11,2025-12-16,San Antonio Spurs at New York Knicks,3070000,2025-12-16_NYK_SAS
38,2026-04-14,Miami Heat at Charlotte Hornets,3070000,2026-04-14_CHA_MIA
55,2026-05-04,NBA Knicks–Sixers,6300000,2026-05-04_
65,2026-05-20,Western Conference Finals Game 2,7770000,2026-05-20_
69,2026-06-03,NBA Finals Game 1,16930000,2026-06-03_
70,2026-06-05,NBA Finals Game 2,16430000,2026-06-05_
71,2026-06-08,NBA Finals Game 3,23790000,2026-06-08_
72,2026-06-10,NBA Finals Game 4,20940000,2026-06-10_
73,2026-06-13,NBA Finals Game 5,24540000,2026-06-13_


In [57]:
playoff_schedule[
    playoff_schedule["date"].isin([
        pd.to_datetime("2026-05-20").date(),
        pd.to_datetime("2026-06-03").date(),
        pd.to_datetime("2026-06-05").date(),
        pd.to_datetime("2026-06-08").date(),
        pd.to_datetime("2026-06-10").date(),
        pd.to_datetime("2026-06-13").date()
    ])
][
    ["date", "Visitor/Neutral", "Home/Neutral", "match_key"]
].sort_values("date")

,date,Visitor/Neutral,Home/Neutral,match_key
71,2026-05-20,San Antonio Spurs,Oklahoma City Thunder,2026-05-20_OKC_SAS
80,2026-06-03,New York Knicks,San Antonio Spurs,2026-06-03_NYK_SAS
81,2026-06-05,New York Knicks,San Antonio Spurs,2026-06-05_NYK_SAS
82,2026-06-08,San Antonio Spurs,New York Knicks,2026-06-08_NYK_SAS
83,2026-06-10,San Antonio Spurs,New York Knicks,2026-06-10_NYK_SAS
84,2026-06-13,New York Knicks,San Antonio Spurs,2026-06-13_NYK_SAS


In [59]:
# Start fresh
ratings_corrected = ratings_2025_26.copy()

# Create a team key from the teams we extracted
ratings_corrected["team_key"] = ratings_corrected["teams_found"].apply(
    lambda teams: "_".join(sorted(teams)) if len(teams) == 2 else ""
)

# Create the initial match key
ratings_corrected["match_key"] = (
    ratings_corrected["date"].astype(str)
    + "_"
    + ratings_corrected["team_key"]
)

# Check the problematic rows
display(
    ratings_corrected[
        ratings_corrected["team_key"] == ""
    ][
        ["date", "telecast", "teams_found", "team_key", "match_key"]
    ]
)

,date,telecast,teams_found,team_key,match_key
55,2026-05-04,NBA Knicks–Sixers,[NYK],,2026-05-04_
65,2026-05-20,Western Conference Finals Game 2,[],,2026-05-20_
69,2026-06-03,NBA Finals Game 1,[],,2026-06-03_
70,2026-06-05,NBA Finals Game 2,[],,2026-06-05_
71,2026-06-08,NBA Finals Game 3,[],,2026-06-08_
72,2026-06-10,NBA Finals Game 4,[],,2026-06-10_
73,2026-06-13,NBA Finals Game 5,[],,2026-06-13_


In [60]:
manual_keys = {
    # December games missing from the original schedule
    "2025-12-09": "2025-12-09_NYK_TOR",
    "2025-12-16": "2025-12-16_NYK_SAS",

    # Play-In game
    "2026-04-14": "2026-04-14_CHA_MIA",

    # "Sixers" wasn't recognized by team_map
    "2026-05-04": "2026-05-04_NYK_PHI",

    # Western Conference Finals Game 2
    "2026-05-20": "2026-05-20_OKC_SAS",

    # NBA Finals — Knicks vs Spurs
    "2026-06-03": "2026-06-03_NYK_SAS",
    "2026-06-05": "2026-06-05_NYK_SAS",
    "2026-06-08": "2026-06-08_NYK_SAS",
    "2026-06-10": "2026-06-10_NYK_SAS",
    "2026-06-13": "2026-06-13_NYK_SAS"
}

for date_str, key in manual_keys.items():
    date_value = pd.to_datetime(date_str).date()
    
    ratings_corrected.loc[
        ratings_corrected["date"] == date_value,
        "match_key"
    ] = key

In [61]:
display(
    ratings_corrected[
        ratings_corrected["date"].isin(
            pd.to_datetime(list(manual_keys.keys())).date
        )
    ][
        ["date", "telecast", "teams_found", "match_key"]
    ]
)

,date,telecast,teams_found,match_key
10,2025-12-09,New York Knicks at Toronto Raptors,"[NYK, TOR]",2025-12-09_NYK_TOR
11,2025-12-16,San Antonio Spurs at New York Knicks,"[NYK, SAS]",2025-12-16_NYK_SAS
38,2026-04-14,Miami Heat at Charlotte Hornets,"[CHA, MIA]",2026-04-14_CHA_MIA
55,2026-05-04,NBA Knicks–Sixers,[NYK],2026-05-04_
65,2026-05-20,Western Conference Finals Game 2,[],2026-05-20_
69,2026-06-03,NBA Finals Game 1,[],2026-06-03_
70,2026-06-05,NBA Finals Game 2,[],2026-06-05_
71,2026-06-08,NBA Finals Game 3,[],2026-06-08_
72,2026-06-10,NBA Finals Game 4,[],2026-06-10_
73,2026-06-13,NBA Finals Game 5,[],2026-06-13_


In [62]:
# Directly assign the correct match keys using the row indices
ratings_corrected.loc[55, "match_key"] = "2026-05-04_NYK_PHI"
ratings_corrected.loc[65, "match_key"] = "2026-05-20_OKC_SAS"
ratings_corrected.loc[69, "match_key"] = "2026-06-03_NYK_SAS"
ratings_corrected.loc[70, "match_key"] = "2026-06-05_NYK_SAS"
ratings_corrected.loc[71, "match_key"] = "2026-06-08_NYK_SAS"
ratings_corrected.loc[72, "match_key"] = "2026-06-10_NYK_SAS"
ratings_corrected.loc[73, "match_key"] = "2026-06-13_NYK_SAS"

# Verify
display(
    ratings_corrected.loc[
        [55, 65, 69, 70, 71, 72, 73],
        ["date", "telecast", "teams_found", "match_key"]
    ]
)

,date,telecast,teams_found,match_key
55,2026-05-04,NBA Knicks–Sixers,[NYK],2026-05-04_NYK_PHI
65,2026-05-20,Western Conference Finals Game 2,[],2026-05-20_OKC_SAS
69,2026-06-03,NBA Finals Game 1,[],2026-06-03_NYK_SAS
70,2026-06-05,NBA Finals Game 2,[],2026-06-05_NYK_SAS
71,2026-06-08,NBA Finals Game 3,[],2026-06-08_NYK_SAS
72,2026-06-10,NBA Finals Game 4,[],2026-06-10_NYK_SAS
73,2026-06-13,NBA Finals Game 5,[],2026-06-13_NYK_SAS


In [63]:
ratings_final = ratings_corrected.merge(
    master_schedule[
        [
            "match_key",
            "game_id",
            "season_type",
            "tip_et",
            "away_team",
            "home_team",
            "national_tv"
        ]
    ],
    on="match_key",
    how="left",
    indicator=True
)

print("Total ratings:", len(ratings_final))
print("Matched:", (ratings_final["_merge"] == "both").sum())
print("Unmatched:", (ratings_final["_merge"] == "left_only").sum())

display(
    ratings_final[
        ratings_final["_merge"] == "left_only"
    ][
        ["date", "telecast", "figure", "match_key"]
    ]
)

Total ratings: 74
Matched: 71
Unmatched: 3


,date,telecast,figure,match_key
10,2025-12-09,New York Knicks at Toronto Raptors,1140000,2025-12-09_NYK_TOR
11,2025-12-16,San Antonio Spurs at New York Knicks,3070000,2025-12-16_NYK_SAS
38,2026-04-14,Miami Heat at Charlotte Hornets,3070000,2026-04-14_CHA_MIA


In [64]:
missing_games = pd.DataFrame([
    {
        "date": pd.to_datetime("2025-12-09").date(),
        "match_key": "2025-12-09_NYK_TOR",
        "game_id": pd.NA,
        "season_type": "Regular Season",
        "tip_et": pd.NA,
        "away_team": "NYK (Knicks)",
        "home_team": "TOR (Raptors)",
        "national_tv": "Prime Video"
    },
    {
        "date": pd.to_datetime("2025-12-16").date(),
        "match_key": "2025-12-16_NYK_SAS",
        "game_id": pd.NA,
        "season_type": "Regular Season",
        "tip_et": pd.NA,
        "away_team": "SAS (Spurs)",
        "home_team": "NYK (Knicks)",
        "national_tv": "Prime Video"
    },
    {
        "date": pd.to_datetime("2026-04-14").date(),
        "match_key": "2026-04-14_CHA_MIA",
        "game_id": pd.NA,
        "season_type": "Play-In",
        "tip_et": pd.NA,
        "away_team": "MIA (Heat)",
        "home_team": "CHA (Hornets)",
        "national_tv": pd.NA
    }
])

master_schedule = pd.concat(
    [master_schedule, missing_games],
    ignore_index=True
)

print("Master schedule rows:", len(master_schedule))

Master schedule rows: 1359


In [65]:
ratings_final = ratings_corrected.merge(
    master_schedule[
        [
            "match_key",
            "game_id",
            "season_type",
            "tip_et",
            "away_team",
            "home_team",
            "national_tv"
        ]
    ],
    on="match_key",
    how="left",
    indicator=True
)

print("Total ratings:", len(ratings_final))
print("Matched:", (ratings_final["_merge"] == "both").sum())
print("Unmatched:", (ratings_final["_merge"] == "left_only").sum())

Total ratings: 74
Matched: 74
Unmatched: 0


In [66]:
# Basic structure
print("Rows:", len(ratings_final))
print("Columns:")
print(ratings_final.columns.tolist())

# Check for duplicate match keys
print("\nDuplicate match keys:", ratings_final["match_key"].duplicated().sum())

# Check for missing key fields
print("\nMissing values:")
print(
    ratings_final[
        ["date", "telecast", "figure", "match_key",
         "season_type", "away_team", "home_team"]
    ].isna().sum()
)

# Check date range
print("\nDate range:")
print(ratings_final["date"].min(), "to", ratings_final["date"].max())

# Check season type
print("\nSeason types:")
print(ratings_final["season_type"].value_counts(dropna=False))

# Preview
display(
    ratings_final[
        [
            "date",
            "telecast",
            "figure",
            "season_type",
            "away_team",
            "home_team",
            "national_tv"
        ]
    ].head(10)
)

Rows: 74
Columns:
['date', 'telecast', 'figure', 'teams_found', 'team_key', 'match_key', 'game_id', 'season_type', 'tip_et', 'away_team', 'home_team', 'national_tv', '_merge']

Duplicate match keys: 0

Missing values:
date           0
telecast       0
figure         0
match_key      0
season_type    0
away_team      0
home_team      0
dtype: int64

Date range:
2025-10-21 00:00:00 to 2026-06-13 00:00:00

Season types:
Regular Season    38
Playoffs          35
Play-In            1
Name: season_type, dtype: int64


,date,telecast,figure,season_type,away_team,home_team,national_tv
0,2025-10-21,NBA Rockets–Thunder,5900000,Regular Season,HOU (Rockets),OKC (Thunder),NBC | Peacock
1,2025-10-22,NBA Cavaliers–Knicks,2170000,Regular Season,CLE (Cavaliers),NYK (Knicks),ESPN
2,2025-10-22,NBA Spurs–Mavericks,2490000,Regular Season,SAS (Spurs),DAL (Mavericks),ESPN
3,2025-10-29,NBA Cavaliers–Celtics,1280000,Regular Season,CLE (Cavaliers),BOS (Celtics),ESPN
4,2025-10-29,NBA Lakers–Timberwolves,1360000,Regular Season,LAL (Lakers),MIN (Timberwolves),ESPN
5,2025-10-31,NBA Celtics–76ers,961000,Regular Season,BOS (Celtics),PHI (76ers),Prime Video
6,2025-10-31,NBA Lakers–Grizzlies,1200000,Regular Season,LAL (Lakers),MEM (Grizzlies),Prime Video
7,2025-11-14,NBA Warriors–Spurs,1430000,Regular Season,GSW (Warriors),SAS (Spurs),Prime Video
8,2025-11-26,Minnesota Timberwolves at Oklahoma City Thunder,2433000,Regular Season,MIN (Timberwolves),OKC (Thunder),ESPN
9,2025-11-26,Houston Rockets at Golden State Warriors,2031000,Regular Season,HOU (Rockets),GSW (Warriors),ESPN


In [67]:
#POTENTIALLY DELETE 
#Create a clean analysis dataset
media_analysis = ratings_final.copy()

# Audience in millions
media_analysis["audience_millions"] = media_analysis["figure"] / 1_000_000

# Calendar variables
media_analysis["year"] = pd.to_datetime(media_analysis["date"]).dt.year
media_analysis["month"] = pd.to_datetime(media_analysis["date"]).dt.month
media_analysis["month_name"] = pd.to_datetime(media_analysis["date"]).dt.month_name()

# Day of week
media_analysis["day_of_week"] = pd.to_datetime(
    media_analysis["date"]
).dt.day_name()

# Flag playoff / postseason games
media_analysis["postseason"] = media_analysis["season_type"].isin(
    ["Playoffs", "Play-In"]
)

# Clean network field
media_analysis["network"] = media_analysis["national_tv"].fillna("Unknown")

# Keep the most useful columns together
media_analysis = media_analysis[
    [
        "date",
        "telecast",
        "figure",
        "audience_millions",
        "season_type",
        "postseason",
        "away_team",
        "home_team",
        "national_tv",
        "network",
        "day_of_week",
        "month_name",
        "year",
        "game_id",
        "match_key"
    ]
].copy()

display(media_analysis.head())

,date,telecast,figure,audience_millions,season_type,postseason,away_team,home_team,national_tv,network,day_of_week,month_name,year,game_id,match_key
0,2025-10-21,NBA Rockets–Thunder,5900000,5.90,Regular Season,False,HOU (Rockets),OKC (Thunder),NBC | Peacock,NBC | Peacock,Tuesday,October,2025,22500001,2025-10-21_HOU_OKC
1,2025-10-22,NBA Cavaliers–Knicks,2170000,2.17,Regular Season,False,CLE (Cavaliers),NYK (Knicks),ESPN,ESPN,Wednesday,October,2025,22500003,2025-10-22_CLE_NYK
2,2025-10-22,NBA Spurs–Mavericks,2490000,2.49,Regular Season,False,SAS (Spurs),DAL (Mavericks),ESPN,ESPN,Wednesday,October,2025,22500004,2025-10-22_DAL_SAS
3,2025-10-29,NBA Cavaliers–Celtics,1280000,1.28,Regular Season,False,CLE (Cavaliers),BOS (Celtics),ESPN,ESPN,Wednesday,October,2025,22500128,2025-10-29_BOS_CLE
4,2025-10-29,NBA Lakers–Timberwolves,1360000,1.36,Regular Season,False,LAL (Lakers),MIN (Timberwolves),ESPN,ESPN,Wednesday,October,2025,22500136,2025-10-29_LAL_MIN


In [68]:
print("Analysis dataset shape:", media_analysis.shape)

print("\nAudience summary:")
display(
    media_analysis["audience_millions"].describe()
)

print("\nAverage audience by season type:")
display(
    media_analysis.groupby("season_type")["audience_millions"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)

Analysis dataset shape: (74, 15)

Audience summary:


count    74.000000
mean      5.175473
std       5.058357
min       0.961000
25%       2.065750
50%       3.470000
75%       6.175000
max      24.540000
Name: audience_millions, dtype: float64


Average audience by season type:


,count,mean,median,min,max
season_type,,,,,
Play-In,1,3.07,3.07,3.07,3.07
Playoffs,35,8.17,6.20,1.16,24.54
Regular Season,38,2.47,2.20,0.96,6.37


In [70]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

tracker_url = "https://www.sportsmediawatch.com/sports-ratings-tracker/?smwnf_league=NBA"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/153.0.0.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,application/xml;"
        "q=0.9,image/avif,image/webp,*/*;q=0.8"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.google.com/"
}

response = requests.get(
    tracker_url,
    headers=headers,
    timeout=30
)

print("Status code:", response.status_code)
print("Response size:", len(response.content))

soup = BeautifulSoup(response.content, "html.parser")

tables = soup.find_all("table")

print("Number of tables:", len(tables))

for i, table in enumerate(tables):
    print(f"\nTABLE {i}")

    rows = table.find_all("tr")

    for row in rows[:6]:
        cells = row.find_all(["th", "td"])
        print([cell.get_text(" ", strip=True) for cell in cells])

    print("Rows found:", len(rows))

Status code: 200
Response size: 570331
Number of tables: 0


In [71]:
# Search the downloaded HTML for recognizable NBA ratings data

html = response.text

search_terms = [
    "Rockets",
    "Thunder",
    "2025-10-21",
    "5900000",
    "sportsmediawatch",
    "smwnf"
]

for term in search_terms:
    print(f"{term}: {html.find(term)}")

Rockets: 147277
Thunder: 149871
2025-10-21: -1
5900000: -1
sportsmediawatch: 675
smwnf: 52791


In [72]:
# Look for embedded JSON / JavaScript data structures

for keyword in [
    "ajax",
    "wp-json",
    "api",
    "json",
    "ratings",
    "tracker",
    "smwnf"
]:
    positions = []
    start = 0

    while True:
        pos = html.lower().find(keyword.lower(), start)

        if pos == -1:
            break

        positions.append(pos)
        start = pos + len(keyword)

        if len(positions) >= 10:
            break

    print(f"\n{keyword}: {positions}")


ajax: [47388, 53735, 53755]

wp-json: [56711, 56821]

api: [47399, 53746, 56660, 78171, 83447, 92412, 92597, 92706, 94949, 169443]

json: [44129, 56714, 56752, 56776, 56824, 61034, 61488, 564146]

ratings: [14509, 42834, 43004, 43162, 43254, 43429, 44276, 44345, 44379, 44555]

tracker: [14517, 43012, 43170, 43437, 44284, 44353, 44387, 44563, 44655, 45084]

smwnf: [52791, 52884, 124931, 125056, 125233, 125286, 125440, 125585, 206471, 206820]


In [73]:
# Inspect the HTML around each AJAX reference

for pos in [47388, 53735, 53755]:
    print("\n" + "=" * 80)
    print("POSITION:", pos)
    print("=" * 80)
    print(html[pos - 1500:pos + 3000])


POSITION: 47388
"item":"https:\/\/www.sportsmediawatch.com\/"},{"@type":"ListItem","position":2,"name":"Sports Ratings Tracker"}]},{"@type":"WebSite","@id":"https:\/\/www.sportsmediawatch.com\/#website","url":"https:\/\/www.sportsmediawatch.com\/","name":"Sports Media Watch","description":"TV ratings, schedules and the latest news","publisher":{"@id":"https:\/\/www.sportsmediawatch.com\/#organization"},"potentialAction":[{"@type":"SearchAction","target":{"@type":"EntryPoint","urlTemplate":"https:\/\/www.sportsmediawatch.com\/?s={search_term_string}"},"query-input":{"@type":"PropertyValueSpecification","valueRequired":true,"valueName":"search_term_string"}}],"inLanguage":"en-US"},{"@type":"Organization","@id":"https:\/\/www.sportsmediawatch.com\/#organization","name":"Sports Media Watch","url":"https:\/\/www.sportsmediawatch.com\/","logo":{"@type":"ImageObject","inLanguage":"en-US","@id":"https:\/\/www.sportsmediawatch.com\/#\/schema\/logo\/image\/","url":"https:\/\/www.sportsmediawatc

In [74]:
# Find all JavaScript files associated with the SMW News Feed plugin

import re

smw_scripts = re.findall(
    r'<script[^>]+src=["\']([^"\']+)["\']',
    html,
    flags=re.IGNORECASE
)

for script in smw_scripts:
    if "smw" in script.lower() or "news-feed" in script.lower():
        print(script)

https://www.sportsmediawatch.com/wp-content/cache/perfmatters/www.sportsmediawatch.com/minify/474aabcfe7e3.smwlb-live.min.js?ver=1.0.4-1789832059
https://www.sportsmediawatch.com/wp-content/cache/perfmatters/www.sportsmediawatch.com/minify/2c714491d738.smwnf-carousel-init.min.js
https://www.sportsmediawatch.com/wp-content/plugins/smw-schedules-manager/js/schedules-viewer.js?ver=1.0.0


In [75]:
# Show every occurrence of the SMW News Feed plugin path

matches = list(
    re.finditer(
        r'smw-news-feed',
        html,
        flags=re.IGNORECASE
    )
)

print("Occurrences:", len(matches))

for match in matches[:20]:
    pos = match.start()
    print("\n---")
    print(html[pos - 300:pos + 500])

Occurrences: 1

---
yle-inline-css */
</style>
<link rel="stylesheet" id="imgareaselect-css" media="all" data-pmdelayedstyle="https://www.sportsmediawatch.com/wp-includes/js/imgareaselect/imgareaselect.css?ver=0.9.8">
<link rel='stylesheet' id='smwnf-style-css' href='https://www.sportsmediawatch.com/wp-content/plugins/smw-news-feed/css/smwnf-style.css?ver=fea006d1625aae34a65b541e67e52474' media='all' />
<link rel='stylesheet' id='swiper-style-css' href='https://cdn.jsdelivr.net/npm/swiper@11/swiper-bundle.min.css?ver=fea006d1625aae34a65b541e67e52474' media='all' />
<link rel='stylesheet' id='flatpickr-style-css' href='https://cdn.jsdelivr.net/npm/flatpickr/dist/flatpickr.min.css?ver=fea006d1625aae34a65b541e67e52474' media='all' />
<link rel='stylesheet' id='schedules-manager-frontend-css' href='https://www.sp


In [76]:
# Download the SMW schedules manager JavaScript and inspect it for API/AJAX endpoints

js_url = "https://www.sportsmediawatch.com/wp-content/plugins/smw-schedules-manager/js/schedules-viewer.js?ver=1.0.0"

js_response = requests.get(js_url, headers=headers, timeout=30)

print("Status:", js_response.status_code)
print("Size:", len(js_response.text))
print()
print(js_response.text[:5000])

Status: 200
Size: 306

jQuery(document).ready(function($) {
    // You can also add an event to smoothly scroll to the top when the "back to top" link is clicked.
    $('.backtotop').click(function(e) {
        e.preventDefault();
        $('html, body').animate({
            scrollTop: 0
        }, 800);
    });
    
    
});



In [77]:
# Inspect the SMW tracker HTML around the remaining smwnf references

for pos in [124931, 125056, 125233, 125286, 125440, 125585, 206471, 206820]:
    print("\n" + "="*100)
    print("POSITION:", pos)
    print("="*100)
    print(html[pos-1000:pos+2500])


POSITION: 124931
    width: 100%; /* Full width */
        padding: 10px 0; /* Space between checkboxes */
        border-bottom: 1px solid var(--border-color, #ececec); /* Separator */
    }

    .mobile-multi-select {
        display: none;
    }
}

/* Remove bottom border from the last checkbox */
.desktop-checkboxes label:last-child {
    border-bottom: none;
}

/* Adjust checkbox spacing */
.desktop-checkboxes input[type="checkbox"] {
    margin-right: 10px; /* Space between checkbox and text */
}

/* Background color adapts to light/dark mode */
@media (prefers-color-scheme: dark) {
    .desktop-checkboxes {
        background-color: #222; /* Dark mode background */
        color: #fff; /* Light text */
        --border-color: #444; /* Dark mode border */
    }
}

@media (prefers-color-scheme: light) {
    .desktop-checkboxes {
       /* background-color: #f8f8f8;*/ /* Light mode background */
        color: #000; /* Dark text */
        --border-color: #ececec; /* Light mode bo

In [78]:
# Inspect the tracker content immediately after the filter form

from bs4 import BeautifulSoup

soup = BeautifulSoup(html, "html.parser")

form = soup.find("form", id="smwnf-filter-form")

print("Form found:", form is not None)

if form:
    # Show the next ~10 elements after the form
    current = form
    for i in range(10):
        current = current.find_next()
        if current:
            print("\n---", i + 1, "---")
            print(
                current.name,
                current.get("id"),
                current.get("class")
            )
            print(current.get_text(" ", strip=True)[:500])

Form found: True

--- 1 ---
div None None
Filter League Date Search All All Leagues College Baseball College Football College Hockey College Lacrosse College Softball College Volleyball College Wrestling F1 Figure Skating Gymnastics Horse Racing IndyCar International Soccer LIV Golf LPGA Little League MCBB MCBB, WCBB MLB MLS NASCAR NBA NFL NHL NHRA NHRA, IndyCar NWSL OLY Other Events PGA TOUR Premier League Tennis Track & Field UFC UFL WCBB WNBA Apply

--- 2 ---
div None None
Filter League Date Search All All Leagues College Baseball College Football College Hockey College Lacrosse College Softball College Volleyball College Wrestling F1 Figure Skating Gymnastics Horse Racing IndyCar International Soccer LIV Golf LPGA Little League MCBB MCBB, WCBB MLB MLS NASCAR NBA NFL NHL NHRA NHRA, IndyCar NWSL OLY Other Events PGA TOUR Premier League Tennis Track & Field UFC UFL WCBB WNBA Apply

--- 3 ---
div None None
Filter League Date Search All

--- 4 ---
span None None
Filter

--- 5 ---
span s

In [79]:
# Inspect the top-level elements immediately following the filter form

current = form

for i in range(15):
    current = current.find_next_sibling()
    if current is None:
        break

    print("\n" + "="*80)
    print("SIBLING", i + 1)
    print("TAG:", current.name)
    print("ID:", current.get("id"))
    print("CLASS:", current.get("class"))
    print("TEXT:", current.get_text(" ", strip=True)[:1000])


SIBLING 1
TAG: script
ID: None
CLASS: None
TEXT: 

SIBLING 2
TAG: div
ID: None
CLASS: ['smwnf-newsfeed']
TEXT: WNBA Prime Video WNBA highs Aug 20, 2026 · Prime Video Thursday’s Fever-Wings WNBA regular season game averaged 1.07 million viewers on Prime Video, marking the streamer’s largest WNBA audience. (Prime carried WNBA games prior to this season, but those were not Nielsen-rated.) The previous high for Prime came two weeks earlier, when the Aces’ buzzer-beating win over the Fever averaged 948,000. Prime had another Fever game on Saturday against the Liberty, figures for which will not be available until Tuesday at the earliest. By Jon Lewis on Aug 24, 2026 at 10:16 AM Tags: WNBA on Prime Video

SIBLING 3
TAG: div
ID: None
CLASS: ['smwnf-newsfeed']
TEXT: WNBA Recent WNBA Aug 16–18, 2026 · ESPN, CBS Tuesday’s Fever-Tempo WNBA regular season game averaged 1.85 million viewers on ESPN, marking the network’s third-largest audience of the season, behind last Sunday’s Fever-Dream game (

In [80]:
# Inspect all form parameters and pagination controls

print("FORM PARAMETERS")
print("=" * 80)

for inp in form.find_all(["input", "select", "button"]):
    print(
        inp.name,
        "| name =", inp.get("name"),
        "| id =", inp.get("id"),
        "| type =", inp.get("type"),
        "| value =", inp.get("value")
    )

print("\n\nPAGINATION / NAVIGATION")
print("=" * 80)

# Look for common pagination/navigation elements
for el in soup.find_all(["a", "button"]):
    text = el.get_text(" ", strip=True)
    href = el.get("href")

    if (
        any(x in text.lower() for x in ["next", "previous", "older", "more", "page"]) 
        or (href and "paged=" in href)
    ):
        print("TEXT:", repr(text), "| HREF:", href)

FORM PARAMETERS
select | name = smwnf_league | id = smwnf-league-select | type = None | value = None
input | name = smwnf_date | id = smwnf-date-filter | type = text | value = 
input | name = smwnf_search | id = smwnf-search-input | type = text | value = 
button | name = None | id = None | type = submit | value = None


PAGINATION / NAVIGATION
TEXT: 'Baltimore Ravens' | HREF: https://www.sportsmediawatch.com/tv-schedules/nfl-tv-schedule/baltimore-ravens/
TEXT: 'Baltimore Orioles' | HREF: https://www.sportsmediawatch.com/tv-schedules/mlb-tv-schedule/baltimore-orioles/
TEXT: 'As previously noted,' | HREF: https://www.sportsmediawatch.com/2026/07/indiana-fever-viewership-highs-with-without-caitlin-clark/
TEXT: 'Load More' | HREF: None
TEXT: 'Next' | HREF: https://www.sportsmediawatch.com/sports-ratings-tracker/comment-page-1/#comments
TEXT: 'More episodes' | HREF: None
TEXT: "1 Troy Aikman's future, JJ Watt's performance, the state of College Gameday, the return of Doc Rivers and more 09/

In [81]:
# Inspect the Load More button and the HTML/JavaScript around it

load_more = soup.find(
    lambda tag: tag.name in ["button", "a", "div"]
    and "Load More" in tag.get_text(" ", strip=True)
)

print("Found:", load_more is not None)

if load_more:
    print("\nELEMENT:")
    print(load_more)

    print("\n\nPARENT:")
    print(load_more.parent)

    print("\n\nHTML AROUND LOAD MORE:")
    print(load_more.parent.parent.prettify()[:10000])


Found: True

ELEMENT:
<div class="jeg_viewport">
<div class="jeg_header_wrapper">
<div class="jeg_header_instagram_wrapper">
</div>
<!-- HEADER -->
<div class="jeg_header full">
<div class="jeg_midbar jeg_container jeg_navbar_wrapper normal">
<div class="container">
<div class="jeg_nav_row">
<div class="jeg_nav_col jeg_nav_left jeg_nav_grow">
<div class="item_wrap jeg_nav_aligncenter">
<div class="jeg_nav_item jeg_logo jeg_desktop_logo">
<div class="site-title">
<a aria-label="Visit Homepage" href="https://www.sportsmediawatch.com/" style="padding: 0px 0px 0px 0px;">
<img alt="Sports Media Watch" class="jeg_logo_img" data-dark-src="https://www.sportsmediawatch.com/wp-content/uploads/2022/04/smwnewlogo333-dark.svg" data-dark-srcset="https://www.sportsmediawatch.com/wp-content/uploads/2022/04/smwnewlogo333-dark.svg 1x,  2x" data-light-src="https://www.sportsmediawatch.com/wp-content/uploads/2022/04/smwnewlogo333-4.svg" data-light-srcset="https://www.sportsmediawatch.com/wp-content/upload

In [82]:
# Find JavaScript/configuration references related to "Load More"

for pattern in ["Load More", "load-more", "loadMore", "smwnf-newsfeed"]:
    print("\n" + "="*80)
    print("PATTERN:", pattern)
    print("="*80)

    matches = list(re.finditer(re.escape(pattern), html, flags=re.IGNORECASE))
    print("Occurrences:", len(matches))

    for m in matches[:10]:
        print("\n--- position", m.start(), "---")
        print(html[m.start()-500:m.start()+1500])


PATTERN: Load More
Occurrences: 1

--- position 270529 ---
v id="loadmorecontainer"></div><div id="smwnf-load-more-wrapper" style="text-align:center; margin-top:20px;"><button id="smwnf-load-more" data-paged="2" data-max="5" data-payload='{"nf_query_args":{"post_type":"newsfeed","post_status":["publish","pending"],"orderby":"date","order":"DESC","posts_per_page":200,"paged":1,"no_found_rows":true,"s":"","suppress_filters":true,"meta_query":[{"key":"_smwnf_league","value":"NBA","compare":"LIKE"}]},"search_term":"","per_page":25,"include_ratings":false}'>Load More</button></div>
                                    
                                                                    </div>
                            </div>


                                            <div data-perfmatters-lazy-element style="height:1000px; width:100%;"><noscript><div id="comments" class="jeg_comments">
                    <h3 class="comments-title">
						Comments                        <span class="co

In [83]:
# Find scripts containing the SMW News Feed load-more code
import base64
import re

scripts = soup.find_all("script")

found = []

for i, script in enumerate(scripts):
    text = script.string or script.get_text()
    
    if "smwnf-load-more" in text or "loadmorecontainer" in text:
        found.append((i, text))

print("Matching scripts:", len(found))

for i, text in found:
    print("\n" + "="*100)
    print("SCRIPT INDEX:", i)
    print("LENGTH:", len(text))
    print("="*100)
    print(text[:12000])

Matching scripts: 0


In [84]:
# Inspect the HTML immediately before the first smwnf-newsfeed
pos = html.find('class="smwnf-newsfeed"')

print("Position:", pos)
print("\nHTML immediately before feed:\n")
print(html[pos-8000:pos])

Position: 216361

HTML immediately before feed:

R</option><option value="NBA"  selected='selected'>NBA</option><option value="NFL" >NFL</option><option value="NHL" >NHL</option><option value="NHRA" >NHRA</option><option value="NHRA, IndyCar" >NHRA, IndyCar</option><option value="NWSL" >NWSL</option><option value="OLY" >OLY</option><option value="Other Events" >Other Events</option><option value="PGA TOUR" >PGA TOUR</option><option value="Premier League" >Premier League</option><option value="Tennis" >Tennis</option><option value="Track &amp; Field" >Track &amp; Field</option><option value="UFC" >UFC</option><option value="UFL" >UFL</option><option value="WCBB" >WCBB</option><option value="WNBA" >WNBA</option></select></div></div><div id="smwnf-date-box" class="smwnf-filter-box" style="display:none; margin-top:10px;"><div style="display:flex; align-items:center; gap:10px;"><input type="text" name="smwnf_date" id="smwnf-date-filter" value="" placeholder="Select a date" style="width:100%

In [85]:
import requests
import json
from bs4 import BeautifulSoup

ajax_url = "https://www.sportsmediawatch.com/wp-admin/admin-ajax.php"

# Get the Load More button from the current NBA tracker page
soup = BeautifulSoup(html, "html.parser")
load_more = soup.find("button", id="smwnf-load-more")

payload = json.loads(load_more["data-payload"])

print("Starting page:", load_more.get("data-paged"))
print("Max pages:", load_more.get("data-max"))
print("Payload keys:", payload.keys())
print("Posts per page:", payload["nf_query_args"]["posts_per_page"])
print("Per page:", payload["per_page"])


Starting page: 2
Max pages: 5
Payload keys: dict_keys(['nf_query_args', 'search_term', 'per_page', 'include_ratings'])
Posts per page: 200
Per page: 25


In [86]:
import json

page = int(load_more["data-paged"])

ajax_data = {
    "action": "smwnf_load_more_news",
    "paged": page,
    "combined": 1,
    "payload": json.dumps(payload)
}

ajax_response = requests.post(
    ajax_url,
    data=ajax_data,
    headers=headers,
    timeout=30
)

print("Status:", ajax_response.status_code)
print("Response length:", len(ajax_response.text))
print("First 500 characters:\n")
print(ajax_response.text[:500])

Status: 200
Response length: 49734
First 500 characters:

{"success":true,"data":{"html":"<div class=\"smwnf-newsfeed\">    <style>\r\n    \/* Liveblog-style timeline feed *\/\r\n    .smwnf-newsfeed {\r\n        position: relative !important;\r\n        margin: 0 !important;\r\n        padding: 0 0 28px 28px !important;\r\n        box-sizing: border-box !important;\r\n    }\r\n\r\n    .smwnf-newsfeed::before {\r\n        content: \"\" !important;\r\n        position: absolute !important;\r\n        left: 9px !important;\r\n        top: 0 !important;\r\


In [87]:
response_json = ajax_response.json()

page2_html = response_json["data"]["html"]
page2_soup = BeautifulSoup(page2_html, "html.parser")

entries = page2_soup.select("div.smwnf-newsfeed")

print("Number of entries returned:", len(entries))

for i, entry in enumerate(entries[:10]):
    print(f"\nENTRY {i}")
    print(entry.get_text(" ", strip=True)[:300])

Number of entries returned: 25

ENTRY 0
WNBA WNBA Playoffs Sep 21–23, 2025 · ABC, ESPN Tuesday’s Fever-Aces WNBA semifinal Game 2 averaged 1.7 million viewers on ESPN, marking the largest second round playoff audience since Comets-Sparks Game 2 on NBC in the 2000 Western Conference Finals, also the most-watched ever on the ESPN family of 

ENTRY 1
WNBA WNBA Playoffs Sep 14–16, 2025 · ABC, ESPN Tuesday’s Dream-Fever first round WNBA playoff Game 2 averaged 1.5 million viewers on ESPN, down 40% from Fever-Sun Game 2 last year, which featured Caitlin Clark (2.5M), but still the fifth-largest WNBA playoff audience since 2000. Viewership trailed o

ENTRY 2
WNBA ESPN/ABC Regular Season Wrap ABC, ESPN WNBA games averaged 1.3 million viewers across ESPN and ABC during the 2025 regular season, up 6% from last year and the most-watched WNBA season ever on the ESPN networks. Keep in mind that the season average is based on the new Nielsen “Big Data + Panel”

ENTRY 3
NBA Local NBA Finals Game 7 Ju

In [88]:
all_entries = []

for page in range(2, int(load_more["data-max"]) + 1):
    ajax_data = {
        "action": "smwnf_load_more_news",
        "paged": page,
        "combined": 1,
        "payload": json.dumps(payload)
    }

    r = requests.post(
        ajax_url,
        data=ajax_data,
        headers=headers,
        timeout=30
    )

    result = r.json()

    if result.get("success") and result["data"].get("html"):
        page_soup = BeautifulSoup(result["data"]["html"], "html.parser")
        page_entries = page_soup.select("div.smwnf-newsfeed")
        all_entries.extend(page_entries)

        print(f"Page {page}: {len(page_entries)} entries")
    else:
        print(f"Page {page}: FAILED")

print("\nTotal entries retrieved:", len(all_entries))

Page 2: 25 entries
Page 3: 25 entries
Page 4: 25 entries
Page 5: 9 entries

Total entries retrieved: 84


In [89]:
nba_entries = []

for entry in all_entries:
    text = entry.get_text(" ", strip=True)
    if "NBA" in text:
        nba_entries.append(text)

print("NBA entries found:", len(nba_entries))

for i, text in enumerate(nba_entries):
    print(f"\n--- NBA ENTRY {i} ---")
    print(text[:500])

NBA entries found: 84

--- NBA ENTRY 0 ---
WNBA WNBA Playoffs Sep 21–23, 2025 · ABC, ESPN Tuesday’s Fever-Aces WNBA semifinal Game 2 averaged 1.7 million viewers on ESPN, marking the largest second round playoff audience since Comets-Sparks Game 2 on NBC in the 2000 Western Conference Finals, also the most-watched ever on the ESPN family of networks. The previous second round high on ESPN was set by Game 1 of the series last Sunday, which drew 1.4 million. For more on the second round numbers, see this article . By Jon Lewis on Sep 27, 2

--- NBA ENTRY 1 ---
WNBA WNBA Playoffs Sep 14–16, 2025 · ABC, ESPN Tuesday’s Dream-Fever first round WNBA playoff Game 2 averaged 1.5 million viewers on ESPN, down 40% from Fever-Sun Game 2 last year, which featured Caitlin Clark (2.5M), but still the fifth-largest WNBA playoff audience since 2000. Viewership trailed only the two Fever-Sun games last year (Game 1 averaged 1.8M) and Games 4 and 5 of the WNBA Finals (1.7 and 2.2M respectively). Game 1 o

In [90]:
nba_entries = []

for entry in all_entries:
    text = entry.get_text(" ", strip=True)

    # The first category label is the first word(s) before the article title.
    # Look specifically for entries whose category begins with "NBA"
    # rather than "WNBA".
    first_text = text[:100]

    if " WNBA " not in first_text and first_text.startswith("NBA"):
        nba_entries.append(text)

print("Actual NBA entries:", len(nba_entries))

for i, text in enumerate(nba_entries):
    print(f"\n--- NBA ENTRY {i} ---")
    print(text[:700])

Actual NBA entries: 51

--- NBA ENTRY 0 ---
NBA Local NBA Finals Game 7 Jun 22, 2025 · ABC This year’s NBA Finals Game 7 averaged a 31.9 rating in Oklahoma City and a 25.1 in Indianapolis, according to TVB. Tulsa ranked third at a 20.8, with Chicago and Cleveland as the top neutral markets with a 10.9. San Antonio (10.5), Milwaukee (10.2) and Minneapolis-St. Paul (10.1) rounded out the top eight. Nationally, the June 22 contest had 7.6 rating and 16.6 million viewers on ABC . By Jon Lewis on Aug 29, 2025 at 8:03 AM Tags: NBA Finals Source: TVB

--- NBA ENTRY 1 ---
NBA NBA Summer League Jul 20, 2025 · ESPN The NBA Summer League, which concluded last week, averaged 255,000 viewers on the ESPN networks — up 27% from last year. The Kings-Hornets title game averaged 420,000 in a post-Sunday Night Baseball window last weekend, up 3% from last year’s Monday night title game. By Jon Lewis on Jul 26, 2025 at 4:27 PM Tags: NBA on ESPN Source: ESPN

--- NBA ENTRY 2 ---
NBA Summer League Jul 10, 2

In [91]:
# Inspect the structure of a multi-game NBA article

entry = all_entries[40]

print(entry.prettify()[:12000])

<div class="smwnf-newsfeed">
 <article class="smwnf-news-item smwnf-liveblog-style-item" id="post-1330629">
  <header class="smwnf-news-header">
   <div class="smwnf-item-type">
    NBA
   </div>
   <div class="smwnf-news-logo">
    <img alt="" loading="lazy" src="https://www.sportsmediawatch.com/wp-content/uploads/2018/03/nbaplayofflogo.png"/>
   </div>
   <div class="smwnf-news-meta">
    <div class="smwnf-title-text" data-league="NBA" data-title="WCF Games 3 and 4">
     WCF Games 3 and 4
    </div>
    <div class="smwnf-news-subline">
     May 24–26, 2025 · ABC, ESPN, ESPN2
    </div>
   </div>
  </header>
  <div class="smwnf-news-content">
   <p>
    Monday’s Thunder-Timberwolves NBA Western Conference Finals Game 4 averaged a 3.4 rating and 6.53 million viewers across ESPN (3.2, 6.16M) and an ESPN2 “InsightCast” (0.22, 378K), down 7 and 5 percent respectively from Timberwolves-Mavericks on TNT and truTV lat year (3.7, 6.87M), and easily the highest rated and most-watched game of 

In [92]:
# Identify the nationally televised regular-season games
# that do NOT currently have a viewership figure.

national_games = master_schedule[
    (master_schedule["season_type"] == "Regular Season") &
    (master_schedule["national_tv"].notna())
].copy()

known_keys = set(media_analysis["match_key"])

missing_games = national_games[
    ~national_games["match_key"].isin(known_keys)
].copy()

missing_games = missing_games[
    [
        "date",
        "match_key",
        "away_team",
        "home_team",
        "national_tv"
    ]
].sort_values("date").reset_index(drop=True)

print("National regular-season games:", len(national_games))
print("Already have ratings:", len(national_games) - len(missing_games))
print("Missing ratings:", len(missing_games))

display(missing_games)

National regular-season games: 299
Already have ratings: 36
Missing ratings: 263


,date,match_key,away_team,home_team,national_tv
0,2025-10-21,2025-10-21_GSW_LAL,GSW (Warriors),LAL (Lakers),NBC | Peacock
1,2025-10-23,2025-10-23_IND_OKC,OKC (Thunder),IND (Pacers),ESPN
2,2025-10-23,2025-10-23_DEN_GSW,DEN (Nuggets),GSW (Warriors),ESPN
3,2025-10-24,2025-10-24_BOS_NYK,BOS (Celtics),NYK (Knicks),Prime Video
4,2025-10-24,2025-10-24_LAL_MIN,MIN (Timberwolves),LAL (Lakers),Prime Video
...,...,...,...,...,...
258,2026-04-08,2026-04-08_LAC_OKC,OKC (Thunder),LAC (Clippers),ESPN
259,2026-04-09,2026-04-09_BOS_NYK,BOS (Celtics),NYK (Knicks),Prime Video
260,2026-04-09,2026-04-09_GSW_LAL,LAL (Lakers),GSW (Warriors),Prime Video
261,2026-04-10,2026-04-10_ATL_CLE,CLE (Cavaliers),ATL (Hawks),Prime Video


In [93]:
additional_viewership = pd.DataFrame([
    # -------------------------
    # October 2025
    # -------------------------
    {
        "date": "2025-10-22",
        "away_team": "SAS",
        "home_team": "DAL",
        "match_key": "2025-10-22_SAS_DAL",
        "national_tv": "ESPN",
        "viewership": 2860000,
        "measurement_type": "peak",
        "measurement_scope": "game",
        "source": "ESPN Press Room",
        "source_url": "https://espnpressroom.com/press-release/espns-third-most-watched-nba-season-opening-wednesday-ever/",
        "notes": "Peak audience"
    },
    {
        "date": "2025-10-22",
        "away_team": "CLE",
        "home_team": "NYK",
        "match_key": "2025-10-22_CLE_NYK",
        "national_tv": "ESPN",
        "viewership": 2840000,
        "measurement_type": "peak",
        "measurement_scope": "game",
        "source": "ESPN Press Room",
        "source_url": "https://espnpressroom.com/press-release/espns-third-most-watched-nba-season-opening-wednesday-ever/",
        "notes": "Peak audience"
    },
    {
        "date": "2025-10-23",
        "away_team": "IND",
        "home_team": "OKC",
        "match_key": "2025-10-23_IND_OKC",
        "national_tv": "ESPN",
        "viewership": 1980000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2025-10-23",
        "away_team": "DEN",
        "home_team": "GSW",
        "match_key": "2025-10-23_DEN_GSW",
        "national_tv": "ESPN",
        "viewership": 2230000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2025-10-23",
        "away_team": "IND",
        "home_team": "OKC",
        "match_key": "2025-10-23_IND_OKC",
        "national_tv": "ESPN",
        "viewership": 2660000,
        "measurement_type": "peak",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Peak audience"
    },

    # -------------------------
    # November 2025
    # -------------------------
    {
        "date": "2025-11-07",
        "away_team": "GSW",
        "home_team": "DEN",
        "match_key": "2025-11-07_GSW_DEN",
        "national_tv": "Prime Video",
        "viewership": 1060000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2025-11-07",
        "away_team": "HOU",
        "home_team": "SAS",
        "match_key": "2025-11-07_HOU_SAS",
        "national_tv": "Prime Video",
        "viewership": 916000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2025-11-12",
        "away_team": "SAS",
        "home_team": "LAL",
        "match_key": "2025-11-12_SAS_LAL",
        "national_tv": "ESPN",
        "viewership": 1560000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2025-11-12",
        "away_team": "MIN",
        "home_team": "NYK",
        "match_key": "2025-11-12_MIN_NYK",
        "national_tv": "ESPN",
        "viewership": 1380000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2025-11-26",
        "away_team": "MIN",
        "home_team": "OKC",
        "match_key": "2025-11-26_MIN_OKC",
        "national_tv": "ESPN",
        "viewership": 2921000,
        "measurement_type": "peak",
        "measurement_scope": "game",
        "source": "ESPN Press Room",
        "source_url": "https://espnpressroom.com/press-release/most-watched-nba-on-espn-thanksgiving-eve-in-six-years/",
        "notes": "Peak audience"
    },
    {
        "date": "2025-11-26",
        "away_team": "HOU",
        "home_team": "GSW",
        "match_key": "2025-11-26_HOU_GSW",
        "national_tv": "ESPN",
        "viewership": 2448000,
        "measurement_type": "peak",
        "measurement_scope": "game",
        "source": "ESPN Press Room",
        "source_url": "https://espnpressroom.com/press-release/most-watched-nba-on-espn-thanksgiving-eve-in-six-years/",
        "notes": "Peak audience"
    },
    {
        "date": "2025-11-26",
        "away_team": "DET",
        "home_team": "BOS",
        "match_key": "2025-11-26_DET_BOS",
        "national_tv": "ESPN",
        "viewership": 3024000,
        "measurement_type": "peak",
        "measurement_scope": "game",
        "source": "ESPN Press Room",
        "source_url": "https://espnpressroom.com/press-release/most-watched-nba-on-espn-thanksgiving-eve-in-six-years/",
        "notes": "Peak audience"
    },

    # -------------------------
    # December 2025
    # -------------------------
    {
        "date": "2025-12-23",
        "away_team": "DEN",
        "home_team": "DAL",
        "match_key": "2025-12-23_DEN_DAL",
        "national_tv": "NBC/Peacock",
        "viewership": 2000000,
        "measurement_type": "combined",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen + Adobe"
    },
    {
        "date": "2025-12-23",
        "away_team": "HOU",
        "home_team": "LAC",
        "match_key": "2025-12-23_HOU_LAC",
        "national_tv": "NBC/Peacock",
        "viewership": 1500000,
        "measurement_type": "combined",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen + Adobe"
    },
    {
        "date": "2025-12-29",
        "away_team": "SAS",
        "home_team": "CLE",
        "match_key": "2025-12-29_SAS_CLE",
        "national_tv": "NBC/Peacock",
        "viewership": 1700000,
        "measurement_type": "combined",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen + Adobe"
    },
    {
        "date": "2025-12-29",
        "away_team": "DAL",
        "home_team": "POR",
        "match_key": "2025-12-29_DAL_POR",
        "national_tv": "NBC/Peacock",
        "viewership": 1200000,
        "measurement_type": "combined",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen + Adobe"
    },
    {
        "date": "2025-12-30",
        "away_team": "PHI",
        "home_team": "MEM",
        "match_key": "2025-12-30_PHI_MEM",
        "national_tv": "NBC",
        "viewership": 1960000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen TV-only"
    },
    {
        "date": "2025-12-30",
        "away_team": "PHI",
        "home_team": "MEM",
        "match_key": "2025-12-30_PHI_MEM",
        "national_tv": "NBC/Peacock",
        "viewership": 2200000,
        "measurement_type": "combined",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen + Adobe"
    },
    {
        "date": "2025-12-30",
        "away_team": "DET",
        "home_team": "LAL",
        "match_key": "2025-12-30_DET_LAL",
        "national_tv": "NBC",
        "viewership": 1720000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen TV-only"
    },
    {
        "date": "2025-12-30",
        "away_team": "DET",
        "home_team": "LAL",
        "match_key": "2025-12-30_DET_LAL",
        "national_tv": "NBC/Peacock",
        "viewership": 2000000,
        "measurement_type": "combined",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen + Adobe"
    },

    # -------------------------
    # January 2026
    # -------------------------
    {
        "date": "2026-01-15",
        "away_team": "MIN",
        "home_team": "HOU",
        "match_key": "2026-01-15_MIN_HOU",
        "national_tv": "ESPN",
        "viewership": 1530000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-01-15",
        "away_team": "OKC",
        "home_team": "HOU",
        "match_key": "2026-01-15_OKC_HOU",
        "national_tv": "Prime Video",
        "viewership": 982000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-01-15",
        "away_team": "NYK",
        "home_team": "GSW",
        "match_key": "2026-01-15_NYK_GSW",
        "national_tv": "Prime Video",
        "viewership": 1270000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-01-19",
        "away_team": "DAL",
        "home_team": "NYK",
        "match_key": "2026-01-19_DAL_NYK",
        "national_tv": "NBC/Peacock",
        "viewership": 2200000,
        "measurement_type": "combined",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen + Adobe"
    },
    {
        "date": "2026-01-19",
        "away_team": "OKC",
        "home_team": "CLE",
        "match_key": "2026-01-19_OKC_CLE",
        "national_tv": "NBC/Peacock",
        "viewership": 1800000,
        "measurement_type": "combined",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen + Adobe"
    },
    {
        "date": "2026-01-19",
        "away_team": "BOS",
        "home_team": "DET",
        "match_key": "2026-01-19_BOS_DET",
        "national_tv": "NBC/Peacock",
        "viewership": 1800000,
        "measurement_type": "combined",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen + Adobe"
    },
    {
        "date": "2026-01-21",
        "away_team": "LAL",
        "home_team": "CLE",
        "match_key": "2026-01-21_LAL_CLE",
        "national_tv": "ESPN",
        "viewership": 1880000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-01-21",
        "away_team": "SAS",
        "home_team": "HOU",
        "match_key": "2026-01-21_SAS_HOU",
        "national_tv": "ESPN",
        "viewership": 1540000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-01-24",
        "away_team": "LAL",
        "home_team": "DAL",
        "match_key": "2026-01-24_LAL_DAL",
        "national_tv": "ABC",
        "viewership": 2600000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-01-24",
        "away_team": "PHI",
        "home_team": "NYK",
        "match_key": "2026-01-24_PHI_NYK",
        "national_tv": "ABC",
        "viewership": 2060000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },

    # -------------------------
    # February 2026
    # -------------------------
    {
        "date": "2026-02-01",
        "away_team": "DAL",
        "home_team": "HOU",
        "match_key": "2026-02-01_DAL_HOU",
        "national_tv": "ABC",
        "viewership": 1930000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-02-01",
        "away_team": "MIL",
        "home_team": "BOS",
        "match_key": "2026-02-01_MIL_BOS",
        "national_tv": "ESPN",
        "viewership": 1510000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-02-01",
        "away_team": "DET",
        "home_team": "GSW",
        "match_key": "2026-02-01_DET_GSW",
        "national_tv": "ESPN",
        "viewership": 1390000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-02-01",
        "away_team": "TOR",
        "home_team": "ORL",
        "match_key": "2026-02-01_TOR_ORL",
        "national_tv": "ESPN",
        "viewership": 998000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-02-01",
        "away_team": "NYK",
        "home_team": "LAL",
        "match_key": "2026-02-01_NYK_LAL",
        "national_tv": "NBC/Peacock",
        "viewership": 5000000,
        "measurement_type": "peak",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Peak audience"
    },
    {
        "date": "2026-02-20",
        "away_team": "HOU",
        "home_team": "NYK",
        "match_key": "2026-02-20_HOU_NYK",
        "national_tv": "ABC",
        "viewership": 1990000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-02-22",
        "away_team": "DEN",
        "home_team": "GSW",
        "match_key": "2026-02-22_DEN_GSW",
        "national_tv": "ABC",
        "viewership": 2950000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Corrected game date: February 22, 2026"
    },
    {
        "date": "2026-02-22",
        "away_team": "CLE",
        "home_team": "OKC",
        "match_key": "2026-02-22_CLE_OKC",
        "national_tv": "ABC",
        "viewership": 2090000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Corrected game date: February 22, 2026"
    },
    {
        "date": "2026-02-22",
        "away_team": "BOS",
        "home_team": "LAL",
        "match_key": "2026-02-22_BOS_LAL",
        "national_tv": "NBC/Peacock",
        "viewership": 4520000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen TV-only"
    },
    {
        "date": "2026-02-22",
        "away_team": "BOS",
        "home_team": "LAL",
        "match_key": "2026-02-22_BOS_LAL",
        "national_tv": "NBC/Peacock/Telemundo",
        "viewership": 5600000,
        "measurement_type": "combined",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen + Adobe / broader platform audience"
    },
    {
        "date": "2026-02-28",
        "away_team": "LAL",
        "home_team": "GSW",
        "match_key": "2026-02-28_LAL_GSW",
        "national_tv": "ABC",
        "viewership": 2180000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-02-28",
        "away_team": "MIN",
        "home_team": "DEN",
        "match_key": "2026-02-28_MIN_DEN",
        "national_tv": "ABC",
        "viewership": 2740000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-02-28",
        "away_team": "SAS",
        "home_team": "NYK",
        "match_key": "2026-02-28_SAS_NYK",
        "national_tv": "ABC",
        "viewership": 2480000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },

    # -------------------------
    # March 2026
    # -------------------------
    {
        "date": "2026-03-01",
        "away_team": "PHI",
        "home_team": "BOS",
        "match_key": "2026-03-01_PHI_BOS",
        "national_tv": "NBC/Peacock",
        "viewership": 3200000,
        "measurement_type": "combined",
        "measurement_scope": "game",
        "source": "NBC Sports",
        "source_url": "https://www.nbcsports.com/pressbox/press-releases/nbas-return-to-nbc-sports-is-a-viewership-triumph",
        "notes": "Total audience"
    },
    {
        "date": "2026-03-07",
        "away_team": "GSW",
        "home_team": "OKC",
        "match_key": "2026-03-07_GSW_OKC",
        "national_tv": "ABC",
        "viewership": 1900000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-03-08",
        "away_team": "BOS",
        "home_team": "CLE",
        "match_key": "2026-03-08_BOS_CLE",
        "national_tv": "ABC",
        "viewership": 2410000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-03-08",
        "away_team": "NYK",
        "home_team": "LAL",
        "match_key": "2026-03-08_NYK_LAL",
        "national_tv": "ABC",
        "viewership": 3340000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-03-08",
        "away_team": "HOU",
        "home_team": "SAS",
        "match_key": "2026-03-08_HOU_SAS",
        "national_tv": "NBC",
        "viewership": 2470000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen TV-only"
    },
    {
        "date": "2026-03-08",
        "away_team": "HOU",
        "home_team": "SAS",
        "match_key": "2026-03-08_HOU_SAS",
        "national_tv": "NBC/Peacock/Telemundo",
        "viewership": 3200000,
        "measurement_type": "combined",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Broader platform audience"
    },
    {
        "date": "2026-03-11",
        "away_team": "CLE",
        "home_team": "ORL",
        "match_key": "2026-03-11_CLE_ORL",
        "national_tv": "ESPN",
        "viewership": 1150000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-03-11",
        "away_team": "HOU",
        "home_team": "DEN",
        "match_key": "2026-03-11_HOU_DEN",
        "national_tv": "ESPN",
        "viewership": 1120000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-03-11",
        "away_team": "SAC",
        "home_team": "LAC",
        "match_key": "2026-03-11_SAC_LAC",
        "national_tv": "ESPN",
        "viewership": 987000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-03-12",
        "away_team": "GSW",
        "home_team": "HOU",
        "match_key": "2026-03-12_GSW_HOU",
        "national_tv": "Prime Video",
        "viewership": 967000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-03-12",
        "away_team": "LAL",
        "home_team": "DEN",
        "match_key": "2026-03-12_LAL_DEN",
        "national_tv": "Prime Video",
        "viewership": 1390000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-03-13",
        "away_team": "DAL",
        "home_team": "BOS",
        "match_key": "2026-03-13_DAL_BOS",
        "national_tv": "ESPN",
        "viewership": 1690000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-03-13",
        "away_team": "LAC",
        "home_team": "SAS",
        "match_key": "2026-03-13_LAC_SAS",
        "national_tv": "ESPN",
        "viewership": 1120000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-03-13",
        "away_team": "OKC",
        "home_team": "NYK",
        "match_key": "2026-03-13_OKC_NYK",
        "national_tv": "ESPN",
        "viewership": 1580000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-03-13",
        "away_team": "CHI",
        "home_team": "MIL",
        "match_key": "2026-03-13_CHI_MIL",
        "national_tv": "ESPN",
        "viewership": 1050000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-03-14",
        "away_team": "DEN",
        "home_team": "LAL",
        "match_key": "2026-03-14_DEN_LAL",
        "national_tv": "ABC",
        "viewership": 2410000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-03-15",
        "away_team": "MIN",
        "home_team": "OKC",
        "match_key": "2026-03-15_MIN_OKC",
        "national_tv": "ABC",
        "viewership": 1980000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-03-22",
        "away_team": "DET",
        "home_team": "MIN",
        "match_key": "2026-03-22_DET_MIN",
        "national_tv": "ABC",
        "viewership": 1590000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-03-25",
        "away_team": "HOU",
        "home_team": "MIN",
        "match_key": "2026-03-25_HOU_MIN",
        "national_tv": "ESPN",
        "viewership": 1240000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-03-25",
        "away_team": "ATL",
        "home_team": "DET",
        "match_key": "2026-03-25_ATL_DET",
        "national_tv": "ESPN",
        "viewership": 1070000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },

    # -------------------------
    # April 2026
    # -------------------------
    {
        "date": "2026-04-02",
        "away_team": "OKC",
        "home_team": "LAL",
        "match_key": "2026-04-02_OKC_LAL",
        "national_tv": "Prime Video",
        "viewership": 1550000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/2026/04/lakers-ratings-post-injuries-mavericks-nbc-easter-sunday/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-04-04",
        "away_team": "SAS",
        "home_team": "DEN",
        "match_key": "2026-04-04_SAS_DEN",
        "national_tv": "Prime Video",
        "viewership": 960000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/2026/04/nba-regular-season-viewership-increase-first-year-rights-deal/",
        "notes": "Most-watched additional Prime Video matinee; medium confidence"
    },
    {
        "date": "2026-04-08",
        "away_team": "POR",
        "home_team": "SAS",
        "match_key": "2026-04-08_POR_SAS",
        "national_tv": "ESPN",
        "viewership": 1230000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    },
    {
        "date": "2026-04-08",
        "away_team": "ATL",
        "home_team": "CLE",
        "match_key": "2026-04-08_ATL_CLE",
        "national_tv": "ESPN",
        "viewership": 1200000,
        "measurement_type": "average",
        "measurement_scope": "game",
        "source": "Sports Media Watch",
        "source_url": "https://www.sportsmediawatch.com/sports-ratings-tracker/",
        "notes": "Nielsen average"
    }
])

additional_viewership["date"] = pd.to_datetime(
    additional_viewership["date"]
).dt.date

print("Rows:", len(additional_viewership))
print("Unique games:", additional_viewership["match_key"].nunique())
print("\nMeasurement types:")
print(additional_viewership["measurement_type"].value_counts())

print("\nScopes:")
print(additional_viewership["measurement_scope"].value_counts())

display(additional_viewership.head(10))

Rows: 67
Unique games: 62

Measurement types:
average     48
combined    12
peak         7
Name: measurement_type, dtype: int64

Scopes:
game    67
Name: measurement_scope, dtype: int64


,date,away_team,home_team,match_key,national_tv,viewership,measurement_type,measurement_scope,source,source_url,notes
0,2025-10-22,SAS,DAL,2025-10-22_SAS_DAL,ESPN,2860000,peak,game,ESPN Press Room,https://espnpressroom.com/press-release/espns-...,Peak audience
1,2025-10-22,CLE,NYK,2025-10-22_CLE_NYK,ESPN,2840000,peak,game,ESPN Press Room,https://espnpressroom.com/press-release/espns-...,Peak audience
2,2025-10-23,IND,OKC,2025-10-23_IND_OKC,ESPN,1980000,average,game,Sports Media Watch,https://www.sportsmediawatch.com/sports-rating...,Nielsen average
3,2025-10-23,DEN,GSW,2025-10-23_DEN_GSW,ESPN,2230000,average,game,Sports Media Watch,https://www.sportsmediawatch.com/sports-rating...,Nielsen average
4,2025-10-23,IND,OKC,2025-10-23_IND_OKC,ESPN,2660000,peak,game,Sports Media Watch,https://www.sportsmediawatch.com/sports-rating...,Peak audience
5,2025-11-07,GSW,DEN,2025-11-07_GSW_DEN,Prime Video,1060000,average,game,Sports Media Watch,https://www.sportsmediawatch.com/sports-rating...,Nielsen average
6,2025-11-07,HOU,SAS,2025-11-07_HOU_SAS,Prime Video,916000,average,game,Sports Media Watch,https://www.sportsmediawatch.com/sports-rating...,Nielsen average
7,2025-11-12,SAS,LAL,2025-11-12_SAS_LAL,ESPN,1560000,average,game,Sports Media Watch,https://www.sportsmediawatch.com/sports-rating...,Nielsen average
8,2025-11-12,MIN,NYK,2025-11-12_MIN_NYK,ESPN,1380000,average,game,Sports Media Watch,https://www.sportsmediawatch.com/sports-rating...,Nielsen average
9,2025-11-26,MIN,OKC,2025-11-26_MIN_OKC,ESPN,2921000,peak,game,ESPN Press Room,https://espnpressroom.com/press-release/most-w...,Peak audience


In [94]:
# Check for exact duplicate observations
dupes = additional_viewership[
    additional_viewership.duplicated(
        subset=["match_key", "measurement_type", "viewership"],
        keep=False
    )
].sort_values(["match_key", "measurement_type", "viewership"])

print("Duplicate observations:", len(dupes))
display(dupes)

Duplicate observations: 0


,date,away_team,home_team,match_key,national_tv,viewership,measurement_type,measurement_scope,source,source_url,notes


In [101]:
# Normalize master_schedule team names to 3-letter abbreviations

master_lookup = master_schedule[
    ["date", "away_team", "home_team", "match_key"]
].copy()

# Team names look like "NYK (Knicks)", "LAL (Lakers)", etc.
master_lookup["away_abbr"] = master_lookup["away_team"].str[:3]
master_lookup["home_abbr"] = master_lookup["home_team"].str[:3]

# Make sure dates match
master_lookup["date"] = pd.to_datetime(master_lookup["date"]).dt.date

# Match additional_viewership to the normalized schedule
check = additional_viewership.merge(
    master_lookup,
    left_on=["date", "away_team", "home_team"],
    right_on=["date", "away_abbr", "home_abbr"],
    how="left",
    suffixes=("_additional", "_master")
)

still_unmatched = check[
    check["match_key_master"].isna()
].copy()

print("Additional-viewership observations:", len(additional_viewership))
print("Matched observations:", len(check) - len(still_unmatched))
print("Still unmatched:", len(still_unmatched))

display(
    still_unmatched[
        [
            "date",
            "away_team",
            "home_team",
            "match_key_additional"
        ]
    ]
)

Additional-viewership observations: 67
Matched observations: 42
Still unmatched: 25


KeyError: "['away_team', 'home_team'] not in index"

In [102]:
# Show the 25 observations that still don't match

print("Still unmatched:", len(still_unmatched))

display(
    still_unmatched[
        [
            "date",
            "away_team_additional",
            "home_team_additional",
            "match_key_additional",
            "away_team_master",
            "home_team_master",
            "match_key_master"
        ]
    ]
)

Still unmatched: 25


,date,away_team_additional,home_team_additional,match_key_additional,away_team_master,home_team_master,match_key_master
2,2025-10-23,IND,OKC,2025-10-23_IND_OKC,NaN,NaN,NaN
4,2025-10-23,IND,OKC,2025-10-23_IND_OKC,NaN,NaN,NaN
7,2025-11-12,SAS,LAL,2025-11-12_SAS_LAL,NaN,NaN,NaN
8,2025-11-12,MIN,NYK,2025-11-12_MIN_NYK,NaN,NaN,NaN
14,2025-12-29,SAS,CLE,2025-12-29_SAS_CLE,NaN,NaN,NaN
20,2026-01-15,MIN,HOU,2026-01-15_MIN_HOU,NaN,NaN,NaN
26,2026-01-21,LAL,CLE,2026-01-21_LAL_CLE,NaN,NaN,NaN
27,2026-01-21,SAS,HOU,2026-01-21_SAS_HOU,NaN,NaN,NaN
29,2026-01-24,PHI,NYK,2026-01-24_PHI_NYK,NaN,NaN,NaN
30,2026-02-01,DAL,HOU,2026-02-01_DAL_HOU,NaN,NaN,NaN


In [106]:
# Inspect master_schedule for the dates of the remaining unmatched games

unmatched_dates = sorted(still_unmatched["date"].unique())

master_date_check = master_schedule[
    pd.to_datetime(master_schedule["date"]).dt.date.isin(unmatched_dates)
].copy()

master_date_check = master_date_check[
    ["date", "away_team", "home_team", "match_key", "national_tv"]
].sort_values(["date", "match_key"])

print("Master schedule games on these dates:", len(master_date_check))

display(master_date_check)

Master schedule games on these dates: 105


,date,away_team,home_team,match_key,national_tv
86,2025-10-23,DEN (Nuggets),GSW (Warriors),2025-10-23_DEN_GSW,ESPN
85,2025-10-23,OKC (Thunder),IND (Pacers),2025-10-23_IND_OKC,ESPN
243,2025-11-12,ATL (Hawks),SAC (Kings),2025-11-12_ATL_SAC,NaN
236,2025-11-12,MEM (Grizzlies),BOS (Celtics),2025-11-12_BOS_MEM,NaN
233,2025-11-12,MIL (Bucks),CHA (Hornets),2025-11-12_CHA_MIL,NaN
...,...,...,...,...,...
1187,2026-04-02,CLE (Cavaliers),GSW (Warriors),2026-04-02_CLE_GSW,Prime Video
1185,2026-04-02,MIN (Timberwolves),DET (Pistons),2026-04-02_DET_MIN,NaN
1189,2026-04-02,SAS (Spurs),LAC (Clippers),2026-04-02_LAC_SAS,NaN
1186,2026-04-02,LAL (Lakers),OKC (Thunder),2026-04-02_LAL_OKC,Prime Video


In [107]:
# Validate additional_viewership using the canonical match_key

schedule_keys = set(master_schedule["match_key"])

additional_key_check = additional_viewership[
    ~additional_viewership["match_key"].isin(schedule_keys)
].copy()

print("Additional-viewership observations:", len(additional_viewership))
print("Matched by match_key:", len(additional_viewership) - len(additional_key_check))
print("Still unmatched by match_key:", len(additional_key_check))

display(
    additional_key_check[
        [
            "date",
            "away_team",
            "home_team",
            "match_key",
            "viewership",
            "measurement_type"
        ]
    ]
)

Additional-viewership observations: 67
Matched by match_key: 27
Still unmatched by match_key: 40


,date,away_team,home_team,match_key,viewership,measurement_type
0,2025-10-22,SAS,DAL,2025-10-22_SAS_DAL,2860000,peak
5,2025-11-07,GSW,DEN,2025-11-07_GSW_DEN,1060000,average
7,2025-11-12,SAS,LAL,2025-11-12_SAS_LAL,1560000,average
8,2025-11-12,MIN,NYK,2025-11-12_MIN_NYK,1380000,average
10,2025-11-26,HOU,GSW,2025-11-26_HOU_GSW,2448000,peak
11,2025-11-26,DET,BOS,2025-11-26_DET_BOS,3024000,peak
12,2025-12-23,DEN,DAL,2025-12-23_DEN_DAL,2000000,combined
14,2025-12-29,SAS,CLE,2025-12-29_SAS_CLE,1700000,combined
16,2025-12-30,PHI,MEM,2025-12-30_PHI_MEM,1960000,average
17,2025-12-30,PHI,MEM,2025-12-30_PHI_MEM,2200000,combined


In [109]:
# Rebuild additional_viewership match_key using the same
# canonical alphabetical team ordering as master_schedule

additional_viewership["match_key"] = (
    additional_viewership["date"].astype(str)
    + "_"
    + additional_viewership[["away_team", "home_team"]]
        .apply(lambda row: "_".join(sorted(row)), axis=1)
)

# Validate against master_schedule
schedule_keys = set(master_schedule["match_key"])

additional_key_check = additional_viewership[
    ~additional_viewership["match_key"].isin(schedule_keys)
].copy()

print("Additional-viewership observations:", len(additional_viewership))
print("Matched by canonical match_key:", 
      len(additional_viewership) - len(additional_key_check))
print("Still unmatched:", len(additional_key_check))

display(
    additional_key_check[
        [
            "date",
            "away_team",
            "home_team",
            "match_key",
            "viewership",
            "measurement_type"
        ]
    ]
)

Additional-viewership observations: 67
Matched by canonical match_key: 48
Still unmatched: 19


,date,away_team,home_team,match_key,viewership,measurement_type
7,2025-11-12,SAS,LAL,2025-11-12_LAL_SAS,1560000,average
8,2025-11-12,MIN,NYK,2025-11-12_MIN_NYK,1380000,average
20,2026-01-15,MIN,HOU,2026-01-15_HOU_MIN,1530000,average
26,2026-01-21,LAL,CLE,2026-01-21_CLE_LAL,1880000,average
27,2026-01-21,SAS,HOU,2026-01-21_HOU_SAS,1540000,average
30,2026-02-01,DAL,HOU,2026-02-01_DAL_HOU,1930000,average
32,2026-02-01,DET,GSW,2026-02-01_DET_GSW,1390000,average
33,2026-02-01,TOR,ORL,2026-02-01_ORL_TOR,998000,average
35,2026-02-20,HOU,NYK,2026-02-20_HOU_NYK,1990000,average
41,2026-02-28,MIN,DEN,2026-02-28_DEN_MIN,2740000,average


In [126]:
date_corrections = {
    7:  "2025-11-05",  # SAS-LAL
    8:  "2025-11-05",  # MIN-NYK
    20: "2026-01-16",  # MIN-HOU
    26: "2026-01-28",  # LAL-CLE
    27: "2026-01-20",  # SAS-HOU
    30: "2026-01-31",  # DAL-HOU
    32: "2026-01-30",  # DET-GSW
    33: "2026-01-30",  # TOR-ORL
    35: "2026-02-21",  # HOU-NYK
    41: "2026-03-01",  # MIN-DEN
    42: "2026-03-01",  # SAS-NYK
    51: "2026-03-14",  # SAC-LAC
    52: "2026-03-05",  # GSW-HOU
    53: "2026-03-05",  # LAL-DEN
    54: "2026-03-06",  # DAL-BOS
    55: "2026-03-06",  # LAC-SAS
    56: "2026-03-04",  # OKC-NYK
    57: "2026-02-03",  # CHI-MIL
    60: "2026-03-28",  # DET-MIN
}

for idx, correct_date in date_corrections.items():
    additional_viewership.loc[idx, "date"] = correct_date

print("Corrected dates:")
print(additional_viewership.loc[
    list(date_corrections.keys()),
    ["date", "away_team", "home_team", "viewership", "measurement_type"]
])

Corrected dates:
          date away_team home_team  viewership measurement_type
7   2025-11-05       SAS       LAL     1560000          average
8   2025-11-05       MIN       NYK     1380000          average
20  2026-01-16       MIN       HOU     1530000          average
26  2026-01-28       LAL       CLE     1880000          average
27  2026-01-20       SAS       HOU     1540000          average
30  2026-01-31       DAL       HOU     1930000          average
32  2026-01-30       DET       GSW     1390000          average
33  2026-01-30       TOR       ORL      998000          average
35  2026-02-21       HOU       NYK     1990000          average
41  2026-03-01       MIN       DEN     2740000          average
42  2026-03-01       SAS       NYK     2480000          average
51  2026-03-14       SAC       LAC      987000          average
52  2026-03-05       GSW       HOU      967000          average
53  2026-03-05       LAL       DEN     1390000          average
54  2026-03-06       DA

In [127]:
additional_viewership["match_key"] = (
    additional_viewership["date"].astype(str)
    + "_"
    + additional_viewership[["away_team", "home_team"]]
        .apply(lambda row: "_".join(sorted(row)), axis=1)
)

master_keys = set(master_schedule["match_key"])

matched = additional_viewership[
    additional_viewership["match_key"].isin(master_keys)
]

unmatched = additional_viewership[
    ~additional_viewership["match_key"].isin(master_keys)
]

print(f"Additional-viewership observations: {len(additional_viewership)}")
print(f"Matched by canonical match_key: {len(matched)}")
print(f"Still unmatched: {len(unmatched)}")

if len(unmatched) > 0:
    display(
        unmatched[
            ["date", "away_team", "home_team",
             "match_key", "viewership", "measurement_type"]
        ].sort_values(["date", "away_team", "home_team"])
    )

Additional-viewership observations: 67
Matched by canonical match_key: 67
Still unmatched: 0


In [125]:
master_schedule.iloc[240]

date                          2025-11-12
match_key             2025-11-12_GSW_SAS
game_id                         22500220
season_type               Regular Season
tip_et         2025-11-12T20:00:00-05:00
away_team                 GSW (Warriors)
home_team                    SAS (Spurs)
national_tv                          NaN
Name: 240, dtype: object

In [95]:
# Validate that every additional-viewership game exists in master_schedule
schedule_keys = set(master_schedule["match_key"])

unmatched_additional = additional_viewership[
    ~additional_viewership["match_key"].isin(schedule_keys)
].sort_values("match_key")

print("Games not found in master_schedule:", len(unmatched_additional))
display(unmatched_additional)

Games not found in master_schedule: 40


,date,away_team,home_team,match_key,national_tv,viewership,measurement_type,measurement_scope,source,source_url,notes
0,2025-10-22,SAS,DAL,2025-10-22_SAS_DAL,ESPN,2860000,peak,game,ESPN Press Room,https://espnpressroom.com/press-release/espns-...,Peak audience
5,2025-11-07,GSW,DEN,2025-11-07_GSW_DEN,Prime Video,1060000,average,game,Sports Media Watch,https://www.sportsmediawatch.com/sports-rating...,Nielsen average
8,2025-11-12,MIN,NYK,2025-11-12_MIN_NYK,ESPN,1380000,average,game,Sports Media Watch,https://www.sportsmediawatch.com/sports-rating...,Nielsen average
7,2025-11-12,SAS,LAL,2025-11-12_SAS_LAL,ESPN,1560000,average,game,Sports Media Watch,https://www.sportsmediawatch.com/sports-rating...,Nielsen average
11,2025-11-26,DET,BOS,2025-11-26_DET_BOS,ESPN,3024000,peak,game,ESPN Press Room,https://espnpressroom.com/press-release/most-w...,Peak audience
10,2025-11-26,HOU,GSW,2025-11-26_HOU_GSW,ESPN,2448000,peak,game,ESPN Press Room,https://espnpressroom.com/press-release/most-w...,Peak audience
12,2025-12-23,DEN,DAL,2025-12-23_DEN_DAL,NBC/Peacock,2000000,combined,game,Sports Media Watch,https://www.sportsmediawatch.com/sports-rating...,Nielsen + Adobe
14,2025-12-29,SAS,CLE,2025-12-29_SAS_CLE,NBC/Peacock,1700000,combined,game,Sports Media Watch,https://www.sportsmediawatch.com/sports-rating...,Nielsen + Adobe
17,2025-12-30,PHI,MEM,2025-12-30_PHI_MEM,NBC/Peacock,2200000,combined,game,Sports Media Watch,https://www.sportsmediawatch.com/sports-rating...,Nielsen + Adobe
16,2025-12-30,PHI,MEM,2025-12-30_PHI_MEM,NBC,1960000,average,game,Sports Media Watch,https://www.sportsmediawatch.com/sports-rating...,Nielsen TV-only


In [98]:
print(master_schedule.columns.tolist())
print("\nShape:", master_schedule.shape)

display(master_schedule.head())

['date', 'match_key', 'game_id', 'season_type', 'tip_et', 'away_team', 'home_team', 'national_tv']

Shape: (1359, 8)


,date,match_key,game_id,season_type,tip_et,away_team,home_team,national_tv
0,2025-10-02,2025-10-02_NYK_PHI,12500008,Preseason,2025-10-02T11:00:00-05:00,PHI (76ers),NYK (Knicks),NBA TV
1,2025-10-03,2025-10-03_LAL_PHX,12500001,Preseason,2025-10-03T21:00:00-05:00,PHX (Suns),LAL (Lakers),NBA TV
2,2025-10-03,2025-10-03_MEL_NOP,12500009,Preseason,2025-10-03T04:30:00-05:00,MEL (United),NOP (Pelicans),NBA TV
3,2025-10-04,2025-10-04_NYK_PHI,12500010,Preseason,2025-10-04T10:00:00-05:00,NYK (Knicks),PHI (76ers),NBA TV
4,2025-10-04,2025-10-04_NOP_SEM,12500011,Preseason,2025-10-04T22:00:00-05:00,SEM (Phoenix),NOP (Pelicans),NBA TV


In [96]:
# Identify games already represented in media_analysis
existing_keys = set(media_analysis["match_key"])

overlap = additional_viewership[
    additional_viewership["match_key"].isin(existing_keys)
].sort_values(["match_key", "measurement_type"])

print("Additional observations involving games already in media_analysis:", len(overlap))
display(overlap)

Additional observations involving games already in media_analysis: 7


,date,away_team,home_team,match_key,national_tv,viewership,measurement_type,measurement_scope,source,source_url,notes
1,2025-10-22,CLE,NYK,2025-10-22_CLE_NYK,ESPN,2840000,peak,game,ESPN Press Room,https://espnpressroom.com/press-release/espns-...,Peak audience
9,2025-11-26,MIN,OKC,2025-11-26_MIN_OKC,ESPN,2921000,peak,game,ESPN Press Room,https://espnpressroom.com/press-release/most-w...,Peak audience
38,2026-02-22,BOS,LAL,2026-02-22_BOS_LAL,NBC/Peacock,4520000,average,game,Sports Media Watch,https://www.sportsmediawatch.com/sports-rating...,Nielsen TV-only
39,2026-02-22,BOS,LAL,2026-02-22_BOS_LAL,NBC/Peacock/Telemundo,5600000,combined,game,Sports Media Watch,https://www.sportsmediawatch.com/sports-rating...,Nielsen + Adobe / broader platform audience
44,2026-03-07,GSW,OKC,2026-03-07_GSW_OKC,ABC,1900000,average,game,Sports Media Watch,https://www.sportsmediawatch.com/sports-rating...,Nielsen average
45,2026-03-08,BOS,CLE,2026-03-08_BOS_CLE,ABC,2410000,average,game,Sports Media Watch,https://www.sportsmediawatch.com/sports-rating...,Nielsen average
58,2026-03-14,DEN,LAL,2026-03-14_DEN_LAL,ABC,2410000,average,game,Sports Media Watch,https://www.sportsmediawatch.com/sports-rating...,Nielsen average


In [128]:
print("Shape:", additional_viewership.shape)
print("\nMeasurement types:")
print(additional_viewership["measurement_type"].value_counts())

print("\nMissing values:")
print(additional_viewership.isna().sum())

print("\nDuplicate observations:")
print(
    additional_viewership.duplicated(
        subset=["match_key", "measurement_type", "viewership"]
    ).sum()
)

Shape: (67, 11)

Measurement types:
average     48
combined    12
peak         7
Name: measurement_type, dtype: int64

Missing values:
date                 0
away_team            0
home_team            0
match_key            0
national_tv          0
viewership           0
measurement_type     0
measurement_scope    0
source               0
source_url           0
notes                0
dtype: int64

Duplicate observations:
0


In [129]:
additional_viewership.to_csv(
    "/Users/drewbeal/NBA_Global_Media_Insights/Data/raw/additional_viewership.csv",
    index=False
)

print("Saved:", "/Users/drewbeal/NBA_Global_Media_Insights/Data/raw/additional_viewership.csv")
print("Rows:", len(additional_viewership))

Saved: /Users/drewbeal/NBA_Global_Media_Insights/Data/raw/additional_viewership.csv
Rows: 67


In [ ]:
## TV Media Blog Supplementary Nielsen Data

In [132]:
import pandas as pd

files = {
    "ABC": "NBA_Global_Media_Insights/Data/raw/blog_abc.csv",
    "NBC": "NBA_Global_Media_Insights/Data/raw/blog_nbc.csv",
    "ESPN": "NBA_Global_Media_Insights/Data/raw/blog_espn.csv",
    "Prime": "NBA_Global_Media_Insights/Data/raw/blog_prime.csv"
}

for network, path in files.items():
    df = pd.read_csv(path)
    
    print(f"\n{'='*50}")
    print(f"{network}")
    print(f"Shape: {df.shape}")
    print("Columns:")
    print(df.columns.tolist())
    print("\nFirst 3 rows:")
    display(df.head(3))


ABC
Shape: (23, 5)
Columns:
['Matchup', 'Time (ET)', 'Date', 'Additional network(s)', 'Big Data Viewership (000s)']

First 3 rows:


,Matchup,Time (ET),Date,Additional network(s),Big Data Viewership (000s)
0,Cavaliers/Knicks,12PM,"Thursday, December 25",ESPN + ESPN2,6400 (3818 + 2272 + 300)
1,Spurs/Thunder,2:30PM,"Thursday, December 25",ESPN,6711 (4343 + 2368)
2,Mavericks/Warriors,5PM,"Thursday, December 25",ESPN,6107 (4199+1908)



NBC
Shape: (40, 5)
Columns:
['Matchup', 'Time (ET)', 'Date', 'Big Data Viewership (000s)', 'Big Data + Adobe Analytics streaming (000s)']

First 3 rows:


,Matchup,Time (ET),Date,Big Data Viewership (000s),Big Data + Adobe Analytics streaming (000s)
0,Rockets/Thunder,7:30PM,"Tuesday, October 21",4868,5900.0
1,Warriors/Lakers,10PM,"Tuesday, October 21",3990,5100.0
2,Knicks/Bucks or Clippers/Warriors,8PM/11PM,"Tuesday, October 28",2135,3000.0



ESPN
Shape: (57, 5)
Columns:
['Matchup', 'Time (ET)', 'Date', 'Nielsen Panel Viewership (000s)', 'Nielsen Big Data viewership (000s)']

First 3 rows:


,Matchup,Time (ET),Date,Nielsen Panel Viewership (000s),Nielsen Big Data viewership (000s)
0,Cavaliers/Knicks,7PM,"Wednesday, October 22",NaN,2166.0
1,Spurs/Mavericks,9:30PM,"Wednesday, October 22",NaN,2488.0
2,Thunder/Pacers,7:30PM,"Thursday, October 23",NaN,1975.0



Prime
Shape: (66, 4)
Columns:
['Matchup', 'Time (ET)', 'Date', 'Big Data Viewership (000s)']

First 3 rows:


,Matchup,Time (ET),Date,Big Data Viewership (000s)
0,Celtics/Knicks,7:30PM,"Friday, October 24",1170.0
1,Timberwolves/Lakers,10PM,"Friday, October 24",1330.0
2,Celtics/76ers,7PM,"Friday, October 31",961.0


In [133]:
for network, path in files.items():
    df = pd.read_csv(path)

    print(f"\n{'='*50}")
    print(network)
    print(f"Rows: {len(df)}")
    
    print("\nMissing values:")
    print(df.isna().sum())


ABC
Rows: 23

Missing values:
Matchup                        0
Time (ET)                      0
Date                           0
Additional network(s)         18
Big Data Viewership (000s)     0
dtype: int64

NBC
Rows: 40

Missing values:
Matchup                                         0
Time (ET)                                       0
Date                                            0
Big Data Viewership (000s)                      0
Big Data + Adobe Analytics streaming (000s)    12
dtype: int64

ESPN
Rows: 57

Missing values:
Matchup                               0
Time (ET)                             0
Date                                  0
Nielsen Panel Viewership (000s)       8
Nielsen Big Data viewership (000s)    6
dtype: int64

Prime
Rows: 66

Missing values:
Matchup                        0
Time (ET)                      0
Date                           0
Big Data Viewership (000s)    42
dtype: int64


In [134]:
for network, path in files.items():
    df = pd.read_csv(path)

    print(f"\n{'='*60}")
    print(network)

    if network == "ABC":
        display(df[["Matchup", "Date", "Big Data Viewership (000s)"]].head(10))

    elif network == "NBC":
        display(df[[
            "Matchup",
            "Date",
            "Big Data Viewership (000s)",
            "Big Data + Adobe Analytics streaming (000s)"
        ]].head(10))

    elif network == "ESPN":
        display(df[[
            "Matchup",
            "Date",
            "Nielsen Panel Viewership (000s)",
            "Nielsen Big Data viewership (000s)"
        ]].head(10))

    elif network == "Prime":
        display(df[[
            "Matchup",
            "Date",
            "Big Data Viewership (000s)"
        ]].head(10))


ABC


,Matchup,Date,Big Data Viewership (000s)
0,Cavaliers/Knicks,"Thursday, December 25",6400 (3818 + 2272 + 300)
1,Spurs/Thunder,"Thursday, December 25",6711 (4343 + 2368)
2,Mavericks/Warriors,"Thursday, December 25",6107 (4199+1908)
3,Rockets/Lakers,"Thursday, December 25",5349 (3450+ 1899)
4,Timberwolves/Nuggets,"Thursday, December 25",3610 (2166+1444)
5,Knicks/76ers,"Saturday, January 24",2057
6,Lakers/Mavericks,"Saturday, January 24",2595
7,Mavericks/Rockets,"Saturday, January 31",1933
8,Rockets/Thunder,"Saturday, February 7",1799
9,Warriors/Lakers,"Saturday, February 7",2473



NBC


,Matchup,Date,Big Data Viewership (000s),Big Data + Adobe Analytics streaming (000s)
0,Rockets/Thunder,"Tuesday, October 21",4868,5900.0
1,Warriors/Lakers,"Tuesday, October 21",3990,5100.0
2,Knicks/Bucks or Clippers/Warriors,"Tuesday, October 28",2135,3000.0
3,Magic/Hawks or Thunder/Clippers,"Tuesday, November 4",1538,2000.0
4,Celtics/76ers or Nuggets/Kings,"Tuesday, November 11",2237,2900.0
5,Grizzlies/Spurs or Suns/Trail Blazers,"Tuesday, November 18",1622,2050.0
6,Magic/76ers or Clippers/Lakers,"Tuesday, November 25",1860,2700.0
7,Knicks/Celtics or Thunder/Warriors,"Tuesday, December 2",2384,3200.0
8,Nuggets/Mavericks,"Tuesday, December 23",1734,2000.0
9,Rockets/Clippers,"Tuesday, December 23",1298,1500.0



ESPN


,Matchup,Date,Nielsen Panel Viewership (000s),Nielsen Big Data viewership (000s)
0,Cavaliers/Knicks,"Wednesday, October 22",NaN,2166.0
1,Spurs/Mavericks,"Wednesday, October 22",NaN,2488.0
2,Thunder/Pacers,"Thursday, October 23",NaN,1975.0
3,Nuggets/Warriors,"Thursday, October 23",NaN,2228.0
4,Cavaliers/Celtics,"Wednesday, October 29",1167.0,1281.0
5,Lakers/Timberwolves,"Wednesday, October 29",1186.0,1358.0
6,Timberwolves/Knicks,"Wednesday, November 5",1220.0,1377.0
7,Spurs/Lakers,"Wednesday, November 5",1346.0,1559.0
8,Suns/Clippers,"Saturday, November 8",516.0,NaN
9,Magic/Knicks,"Wednesday, November 12",1116.0,1312.0



Prime


,Matchup,Date,Big Data Viewership (000s)
0,Celtics/Knicks,"Friday, October 24",1170.0
1,Timberwolves/Lakers,"Friday, October 24",1330.0
2,Celtics/76ers,"Friday, October 31",961.0
3,Lakers/Grizzlies,"Friday, October 31",1200.0
4,Rockets/Spurs,"Friday, November 7",916.0
5,Warriors/Nuggets,"Friday, November 7",1060.0
6,Heat/Knicks,"Friday, November 14",NaN
7,Warriors/Spurs,"Friday, November 14",1434.0
8,Pacers/Cavaliers,"Friday, November 21",NaN
9,Nuggets/Rockets,"Friday, November 21",NaN


In [135]:
abc = pd.read_csv(files["ABC"])
nbc = pd.read_csv(files["NBC"])
espn = pd.read_csv(files["ESPN"])
prime = pd.read_csv(files["Prime"])

print("ABC:", abc.shape)
print("NBC:", nbc.shape)
print("ESPN:", espn.shape)
print("Prime:", prime.shape)

ABC: (23, 5)
NBC: (40, 5)
ESPN: (57, 5)
Prime: (66, 4)


In [136]:
#ABC
abc_clean = abc.copy()

# Preserve the original source value
abc_clean["original_viewership"] = abc_clean["Big Data Viewership (000s)"]

# Extract the leading numeric value (in thousands)
abc_clean["viewership_000s"] = (
    abc_clean["Big Data Viewership (000s)"]
    .astype(str)
    .str.extract(r"^(\d+(?:\.\d+)?)")[0]
    .astype(float)
)

# Convert to actual viewers
abc_clean["viewership"] = abc_clean["viewership_000s"] * 1000

# Parse the date
abc_clean["date"] = pd.to_datetime(
    abc_clean["Date"] + " 2025",
    format="%A, %B %d %Y"
)

# Add network
abc_clean["network"] = "ABC"

display(
    abc_clean[
        ["network", "date", "Matchup",
         "viewership_000s", "viewership", "original_viewership"]
    ].head(10)
)

,network,date,Matchup,viewership_000s,viewership,original_viewership
0,ABC,2025-12-25,Cavaliers/Knicks,6400.0,6400000.0,6400 (3818 + 2272 + 300)
1,ABC,2025-12-25,Spurs/Thunder,6711.0,6711000.0,6711 (4343 + 2368)
2,ABC,2025-12-25,Mavericks/Warriors,6107.0,6107000.0,6107 (4199+1908)
3,ABC,2025-12-25,Rockets/Lakers,5349.0,5349000.0,5349 (3450+ 1899)
4,ABC,2025-12-25,Timberwolves/Nuggets,3610.0,3610000.0,3610 (2166+1444)
5,ABC,2025-01-24,Knicks/76ers,2057.0,2057000.0,2057
6,ABC,2025-01-24,Lakers/Mavericks,2595.0,2595000.0,2595
7,ABC,2025-01-31,Mavericks/Rockets,1933.0,1933000.0,1933
8,ABC,2025-02-07,Rockets/Thunder,1799.0,1799000.0,1799
9,ABC,2025-02-07,Warriors/Lakers,2473.0,2473000.0,2473


In [138]:
abc_clean["date"] = pd.to_datetime(
    abc_clean["Date"] + " 2025",
    format="%A, %B %d %Y"
)

# January and February entries belong to 2026
abc_clean.loc[
    abc_clean["date"].dt.month.isin([1, 2, 3]),
    "date"
] = abc_clean.loc[
    abc_clean["date"].dt.month.isin([1, 2, 3]),
    "date"
] + pd.DateOffset(years=1)

display(
    abc_clean[
        ["network", "date", "Matchup", "viewership_000s"]
    ]
)

,network,date,Matchup,viewership_000s
0,ABC,2025-12-25,Cavaliers/Knicks,6400.0
1,ABC,2025-12-25,Spurs/Thunder,6711.0
2,ABC,2025-12-25,Mavericks/Warriors,6107.0
3,ABC,2025-12-25,Rockets/Lakers,5349.0
4,ABC,2025-12-25,Timberwolves/Nuggets,3610.0
5,ABC,2026-01-24,Knicks/76ers,2057.0
6,ABC,2026-01-24,Lakers/Mavericks,2595.0
7,ABC,2026-01-31,Mavericks/Rockets,1933.0
8,ABC,2026-02-07,Rockets/Thunder,1799.0
9,ABC,2026-02-07,Warriors/Lakers,2473.0


In [139]:
#NBC
nbc_clean = nbc.copy()

# Preserve original source values
nbc_clean["original_viewership"] = nbc_clean["Big Data Viewership (000s)"]
nbc_clean["original_streaming"] = nbc_clean[
    "Big Data + Adobe Analytics streaming (000s)"
]

# Convert viewership values from thousands to viewers
nbc_clean["viewership_000s"] = pd.to_numeric(
    nbc_clean["Big Data Viewership (000s)"],
    errors="coerce"
)

nbc_clean["streaming_viewership_000s"] = pd.to_numeric(
    nbc_clean["Big Data + Adobe Analytics streaming (000s)"],
    errors="coerce"
)

nbc_clean["viewership"] = nbc_clean["viewership_000s"] * 1000
nbc_clean["streaming_viewership"] = (
    nbc_clean["streaming_viewership_000s"] * 1000
)

# Parse dates as part of the 2025-26 season
nbc_clean["date"] = pd.to_datetime(
    nbc_clean["Date"] + " 2025",
    format="%A, %B %d %Y"
)

nbc_clean.loc[
    nbc_clean["date"].dt.month.isin([1, 2, 3, 4]),
    "date"
] = nbc_clean.loc[
    nbc_clean["date"].dt.month.isin([1, 2, 3, 4]),
    "date"
] + pd.DateOffset(years=1)

nbc_clean["network"] = "NBC"

display(
    nbc_clean[
        ["network", "date", "Matchup",
         "viewership_000s", "streaming_viewership_000s",
         "viewership", "streaming_viewership"]
    ].head(12)
)

,network,date,Matchup,viewership_000s,streaming_viewership_000s,viewership,streaming_viewership
0,NBC,2025-10-21,Rockets/Thunder,4868,5900.0,4868000,5900000.0
1,NBC,2025-10-21,Warriors/Lakers,3990,5100.0,3990000,5100000.0
2,NBC,2025-10-28,Knicks/Bucks or Clippers/Warriors,2135,3000.0,2135000,3000000.0
3,NBC,2025-11-04,Magic/Hawks or Thunder/Clippers,1538,2000.0,1538000,2000000.0
4,NBC,2025-11-11,Celtics/76ers or Nuggets/Kings,2237,2900.0,2237000,2900000.0
5,NBC,2025-11-18,Grizzlies/Spurs or Suns/Trail Blazers,1622,2050.0,1622000,2050000.0
6,NBC,2025-11-25,Magic/76ers or Clippers/Lakers,1860,2700.0,1860000,2700000.0
7,NBC,2025-12-02,Knicks/Celtics or Thunder/Warriors,2384,3200.0,2384000,3200000.0
8,NBC,2025-12-23,Nuggets/Mavericks,1734,2000.0,1734000,2000000.0
9,NBC,2025-12-23,Rockets/Clippers,1298,1500.0,1298000,1500000.0


In [140]:
#ESPN
espn_clean = espn.copy()

# Preserve original source values
espn_clean["original_panel"] = espn_clean[
    "Nielsen Panel Viewership (000s)"
]
espn_clean["original_big_data"] = espn_clean[
    "Nielsen Big Data viewership (000s)"
]

# Convert both measurements from thousands to viewers
espn_clean["panel_viewership_000s"] = pd.to_numeric(
    espn_clean["Nielsen Panel Viewership (000s)"],
    errors="coerce"
)

espn_clean["viewership_000s"] = pd.to_numeric(
    espn_clean["Nielsen Big Data viewership (000s)"],
    errors="coerce"
)

espn_clean["panel_viewership"] = (
    espn_clean["panel_viewership_000s"] * 1000
)

espn_clean["viewership"] = (
    espn_clean["viewership_000s"] * 1000
)

# Parse dates as part of the 2025-26 season
espn_clean["date"] = pd.to_datetime(
    espn_clean["Date"] + " 2025",
    format="%A, %B %d %Y"
)

espn_clean.loc[
    espn_clean["date"].dt.month.isin([1, 2, 3, 4]),
    "date"
] = espn_clean.loc[
    espn_clean["date"].dt.month.isin([1, 2, 3, 4]),
    "date"
] + pd.DateOffset(years=1)

espn_clean["network"] = "ESPN"

display(
    espn_clean[
        ["network", "date", "Matchup",
         "panel_viewership_000s", "viewership_000s",
         "panel_viewership", "viewership"]
    ].head(12)
)

,network,date,Matchup,panel_viewership_000s,viewership_000s,panel_viewership,viewership
0,ESPN,2025-10-22,Cavaliers/Knicks,NaN,2166.0,NaN,2166000.0
1,ESPN,2025-10-22,Spurs/Mavericks,NaN,2488.0,NaN,2488000.0
2,ESPN,2025-10-23,Thunder/Pacers,NaN,1975.0,NaN,1975000.0
3,ESPN,2025-10-23,Nuggets/Warriors,NaN,2228.0,NaN,2228000.0
4,ESPN,2025-10-29,Cavaliers/Celtics,1167.0,1281.0,1167000.0,1281000.0
5,ESPN,2025-10-29,Lakers/Timberwolves,1186.0,1358.0,1186000.0,1358000.0
6,ESPN,2025-11-05,Timberwolves/Knicks,1220.0,1377.0,1220000.0,1377000.0
7,ESPN,2025-11-05,Spurs/Lakers,1346.0,1559.0,1346000.0,1559000.0
8,ESPN,2025-11-08,Suns/Clippers,516.0,NaN,516000.0,NaN
9,ESPN,2025-11-12,Magic/Knicks,1116.0,1312.0,1116000.0,1312000.0


In [141]:
#PRIME VIDEO
prime_clean = prime.copy()

# Preserve original source value
prime_clean["original_viewership"] = prime_clean[
    "Big Data Viewership (000s)"
]

# Convert Big Data from thousands to viewers
prime_clean["viewership_000s"] = pd.to_numeric(
    prime_clean["Big Data Viewership (000s)"],
    errors="coerce"
)

prime_clean["viewership"] = (
    prime_clean["viewership_000s"] * 1000
)

# Parse dates as part of the 2025-26 season
prime_clean["date"] = pd.to_datetime(
    prime_clean["Date"] + " 2025",
    format="%A, %B %d %Y"
)

prime_clean.loc[
    prime_clean["date"].dt.month.isin([1, 2, 3, 4]),
    "date"
] = prime_clean.loc[
    prime_clean["date"].dt.month.isin([1, 2, 3, 4]),
    "date"
] + pd.DateOffset(years=1)

prime_clean["network"] = "Prime Video"

display(
    prime_clean[
        ["network", "date", "Matchup",
         "viewership_000s", "viewership"]
    ].head(15)
)

,network,date,Matchup,viewership_000s,viewership
0,Prime Video,2025-10-24,Celtics/Knicks,1170.0,1170000.0
1,Prime Video,2025-10-24,Timberwolves/Lakers,1330.0,1330000.0
2,Prime Video,2025-10-31,Celtics/76ers,961.0,961000.0
3,Prime Video,2025-10-31,Lakers/Grizzlies,1200.0,1200000.0
4,Prime Video,2025-11-07,Rockets/Spurs,916.0,916000.0
5,Prime Video,2025-11-07,Warriors/Nuggets,1060.0,1060000.0
6,Prime Video,2025-11-14,Heat/Knicks,NaN,NaN
7,Prime Video,2025-11-14,Warriors/Spurs,1434.0,1434000.0
8,Prime Video,2025-11-21,Pacers/Cavaliers,NaN,NaN
9,Prime Video,2025-11-21,Nuggets/Rockets,NaN,NaN


In [142]:
# Combine the cleaned network datasets
dataset2_raw = pd.concat(
    [
        abc_clean,
        nbc_clean,
        espn_clean,
        prime_clean
    ],
    ignore_index=True,
    sort=False
)

print("Shape:", dataset2_raw.shape)
print("\nColumns:")
print(dataset2_raw.columns.tolist())

display(
    dataset2_raw[
        ["network", "date", "Matchup", "viewership"]
    ].head(20)
)

Shape: (186, 20)

Columns:
['Matchup', 'Time (ET)', 'Date', 'Additional network(s)', 'Big Data Viewership (000s)', 'original_viewership', 'viewership_000s', 'viewership', 'date', 'network', 'Big Data + Adobe Analytics streaming (000s)', 'original_streaming', 'streaming_viewership_000s', 'streaming_viewership', 'Nielsen Panel Viewership (000s)', 'Nielsen Big Data viewership (000s)', 'original_panel', 'original_big_data', 'panel_viewership_000s', 'panel_viewership']


,network,date,Matchup,viewership
0,ABC,2025-12-25,Cavaliers/Knicks,6400000.0
1,ABC,2025-12-25,Spurs/Thunder,6711000.0
2,ABC,2025-12-25,Mavericks/Warriors,6107000.0
3,ABC,2025-12-25,Rockets/Lakers,5349000.0
4,ABC,2025-12-25,Timberwolves/Nuggets,3610000.0
5,ABC,2026-01-24,Knicks/76ers,2057000.0
6,ABC,2026-01-24,Lakers/Mavericks,2595000.0
7,ABC,2026-01-31,Mavericks/Rockets,1933000.0
8,ABC,2026-02-07,Rockets/Thunder,1799000.0
9,ABC,2026-02-07,Warriors/Lakers,2473000.0


In [149]:
regional_rows = dataset2_raw[
    dataset2_raw["Matchup"].astype(str).str.contains(" or ", case=False, na=False)
]

print("Regional / multi-game rows:", len(regional_rows))

display(
    regional_rows[
        ["network", "date", "Matchup", "viewership", "streaming_viewership"]
    ].to_string(index=False)
)

Regional / multi-game rows: 17


'network       date                                  Matchup  viewership  streaming_viewership\n    NBC 2025-10-28        Knicks/Bucks or Clippers/Warriors   2135000.0             3000000.0\n    NBC 2025-11-04          Magic/Hawks or Thunder/Clippers   1538000.0             2000000.0\n    NBC 2025-11-11           Celtics/76ers or Nuggets/Kings   2237000.0             2900000.0\n    NBC 2025-11-18    Grizzlies/Spurs or Suns/Trail Blazers   1622000.0             2050000.0\n    NBC 2025-11-25           Magic/76ers or Clippers/Lakers   1860000.0             2700000.0\n    NBC 2025-12-02       Knicks/Celtics or Thunder/Warriors   2384000.0             3200000.0\n    NBC 2026-01-06     Heat/Timberwolves or Mavericks/Kings   1742000.0                   NaN\n    NBC 2026-01-13  Spurs/Thunder or Trail Blazers/Warriors   2042000.0             2800000.0\n    NBC 2026-01-20          Spurs/Rockets or Lakers/Nuggets   2089000.0                   NaN\n    NBC 2026-01-27             Bucks/76ers or Cli

In [150]:
# Split regional/multi-game windows into individual matchup rows
dataset2_split = dataset2_raw.copy()

window_mask = dataset2_split["Matchup"].astype(str).str.contains(
    " or ", case=False, na=False
)

# Split each window into its component games
window_rows = dataset2_split[window_mask].copy()
window_rows["Matchup"] = window_rows["Matchup"].str.split(" or ")

# One row per individual game in each window
window_rows = window_rows.explode("Matchup").reset_index(drop=True)

# Mark these as window-level measurements
window_rows["measurement_scope"] = "window"

# Keep non-window games as they are
game_rows = dataset2_split[~window_mask].copy()
game_rows["measurement_scope"] = "game"

# Combine them back together
dataset2_split = pd.concat(
    [game_rows, window_rows],
    ignore_index=True
)

# Clean up whitespace
dataset2_split["Matchup"] = dataset2_split["Matchup"].str.strip()

print("Original observations:", len(dataset2_raw))
print("Window observations:", window_mask.sum())
print("Rows after splitting:", len(dataset2_split))

print("\nMeasurement scope:")
print(dataset2_split["measurement_scope"].value_counts())

display(
    dataset2_split[
        ["network", "date", "Matchup",
         "viewership", "streaming_viewership",
         "measurement_scope"]
    ][
        dataset2_split["measurement_scope"] == "window"
    ].head(20)
)

Original observations: 186
Window observations: 17
Rows after splitting: 203

Measurement scope:
game      169
window     34
Name: measurement_scope, dtype: int64


,network,date,Matchup,viewership,streaming_viewership,measurement_scope
169,NBC,2025-10-28,Knicks/Bucks,2135000.0,3000000.0,window
170,NBC,2025-10-28,Clippers/Warriors,2135000.0,3000000.0,window
171,NBC,2025-11-04,Magic/Hawks,1538000.0,2000000.0,window
172,NBC,2025-11-04,Thunder/Clippers,1538000.0,2000000.0,window
173,NBC,2025-11-11,Celtics/76ers,2237000.0,2900000.0,window
174,NBC,2025-11-11,Nuggets/Kings,2237000.0,2900000.0,window
175,NBC,2025-11-18,Grizzlies/Spurs,1622000.0,2050000.0,window
176,NBC,2025-11-18,Suns/Trail Blazers,1622000.0,2050000.0,window
177,NBC,2025-11-25,Magic/76ers,1860000.0,2700000.0,window
178,NBC,2025-11-25,Clippers/Lakers,1860000.0,2700000.0,window


In [151]:
team_map = {
    "Atlanta Hawks": "ATL",
    "Hawks": "ATL",
    "Boston Celtics": "BOS",
    "Celtics": "BOS",
    "Brooklyn Nets": "BKN",
    "Nets": "BKN",
    "Charlotte Hornets": "CHA",
    "Hornets": "CHA",
    "Chicago Bulls": "CHI",
    "Bulls": "CHI",
    "Cleveland Cavaliers": "CLE",
    "Cavaliers": "CLE",
    "Dallas Mavericks": "DAL",
    "Mavericks": "DAL",
    "Denver Nuggets": "DEN",
    "Nuggets": "DEN",
    "Detroit Pistons": "DET",
    "Pistons": "DET",
    "Golden State Warriors": "GSW",
    "Warriors": "GSW",
    "Houston Rockets": "HOU",
    "Rockets": "HOU",
    "Indiana Pacers": "IND",
    "Pacers": "IND",
    "Los Angeles Clippers": "LAC",
    "Clippers": "LAC",
    "Los Angeles Lakers": "LAL",
    "Lakers": "LAL",
    "Memphis Grizzlies": "MEM",
    "Grizzlies": "MEM",
    "Miami Heat": "MIA",
    "Heat": "MIA",
    "Milwaukee Bucks": "MIL",
    "Bucks": "MIL",
    "Minnesota Timberwolves": "MIN",
    "Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP",
    "Pelicans": "NOP",
    "New York Knicks": "NYK",
    "Knicks": "NYK",
    "Oklahoma City Thunder": "OKC",
    "Thunder": "OKC",
    "Orlando Magic": "ORL",
    "Magic": "ORL",
    "Philadelphia 76ers": "PHI",
    "76ers": "PHI",
    "Phoenix Suns": "PHX",
    "Suns": "PHX",
    "Portland Trail Blazers": "POR",
    "Trail Blazers": "POR",
    "Sacramento Kings": "SAC",
    "Kings": "SAC",
    "San Antonio Spurs": "SAS",
    "Spurs": "SAS",
    "Toronto Raptors": "TOR",
    "Raptors": "TOR",
    "Utah Jazz": "UTA",
    "Jazz": "UTA",
    "Washington Wizards": "WAS",
    "Wizards": "WAS"
}

# Extract the two teams from each matchup
dataset2_split["team_1"] = dataset2_split["Matchup"].str.split("/").str[0].str.strip()
dataset2_split["team_2"] = dataset2_split["Matchup"].str.split("/").str[1].str.strip()

# Convert full team names to abbreviations
dataset2_split["team_1"] = dataset2_split["team_1"].map(team_map)
dataset2_split["team_2"] = dataset2_split["team_2"].map(team_map)

# Show any team names that failed to map
unmapped = dataset2_split[
    dataset2_split["team_1"].isna() |
    dataset2_split["team_2"].isna()
]

print("Unmapped matchup rows:", len(unmapped))

if len(unmapped) > 0:
    display(
        unmapped[
            ["network", "date", "Matchup", "team_1", "team_2"]
        ]
    )
else:
    print("All matchup team names mapped successfully.")

Unmapped matchup rows: 11


,network,date,Matchup,team_1,team_2
36,NBC,2026-02-15,NBA All Star Game,NaN,NaN
73,ESPN,2026-01-30,Pistons/Warrios,DET,NaN
117,Prime Video,2025-12-09,NBA Cup QF: Heat/Magic,NaN,ORL
118,Prime Video,2025-12-09,NBA Cup QF: Knicks/Raptors,NaN,TOR
119,Prime Video,2025-12-10,NBA Cup QF: Suns/Thunder,NaN,OKC
120,Prime Video,2025-12-10,NBA Cup QF: Spurs/Lakers,NaN,LAL
121,Prime Video,2025-12-13,NBA Cup SF: Knicks/Magic,NaN,ORL
122,Prime Video,2025-12-13,NBA Cup SF: Spurs/Thunder,NaN,OKC
123,Prime Video,2025-12-16,NBA Cup F: Spurs/Knicks,NaN,NYK
131,Prime Video,2026-01-15,Grizzlies/Magic (Germany),MEM,NaN


In [152]:
# Create a cleaned matchup field without altering the original source value
dataset2_split["Matchup_clean"] = dataset2_split["Matchup"]

# Fix known typo
dataset2_split["Matchup_clean"] = dataset2_split["Matchup_clean"].replace(
    "Pistons/Warrios",
    "Pistons/Warriors"
)

# Remove NBA Cup prefixes
dataset2_split["Matchup_clean"] = (
    dataset2_split["Matchup_clean"]
    .str.replace(r"NBA Cup QF: ", "", regex=True)
    .str.replace(r"NBA Cup SF: ", "", regex=True)
    .str.replace(r"NBA Cup F: ", "", regex=True)
)

# Remove international-location suffixes
dataset2_split["Matchup_clean"] = (
    dataset2_split["Matchup_clean"]
    .str.replace(r" \(Germany\)", "", regex=True)
    .str.replace(r" \(London\)", "", regex=True)
)

# Extract teams again
dataset2_split["team_1"] = (
    dataset2_split["Matchup_clean"]
    .str.split("/")
    .str[0]
    .str.strip()
)

dataset2_split["team_2"] = (
    dataset2_split["Matchup_clean"]
    .str.split("/")
    .str[1]
    .str.strip()
)

# Map to abbreviations
dataset2_split["team_1"] = dataset2_split["team_1"].map(team_map)
dataset2_split["team_2"] = dataset2_split["team_2"].map(team_map)

# Identify anything still unmapped
unmapped = dataset2_split[
    dataset2_split["team_1"].isna() |
    dataset2_split["team_2"].isna()
]

print("Unmapped matchup rows:", len(unmapped))

display(
    unmapped[
        ["network", "date", "Matchup",
         "Matchup_clean", "team_1", "team_2"]
    ]
)

Unmapped matchup rows: 1


,network,date,Matchup,Matchup_clean,team_1,team_2
36,NBC,2026-02-15,NBA All Star Game,NBA All Star Game,NaN,NaN


In [153]:
dataset2_split = dataset2_split[
    dataset2_split["Matchup"] != "NBA All Star Game"
].copy()

print("Rows after removing All-Star Game:", len(dataset2_split))
print("Unmapped team rows:",
      dataset2_split["team_1"].isna().sum() +
      dataset2_split["team_2"].isna().sum())

Rows after removing All-Star Game: 202
Unmapped team rows: 0


In [154]:
dataset2_split["match_key"] = (
    dataset2_split["date"].dt.strftime("%Y-%m-%d")
    + "_"
    + dataset2_split[["team_1", "team_2"]]
        .apply(lambda x: "_".join(sorted(x)), axis=1)
)

dataset2_schedule_check = dataset2_split.merge(
    master_schedule[
        ["match_key", "away_team", "home_team", "game_id",
         "season_type", "national_tv"]
    ],
    on="match_key",
    how="left",
    indicator=True
)

print("Dataset 2 rows:", len(dataset2_split))
print("Matched to master schedule:",
      (dataset2_schedule_check["_merge"] == "both").sum())
print("Unmatched:",
      (dataset2_schedule_check["_merge"] == "left_only").sum())


Dataset 2 rows: 202
Matched to master schedule: 196
Unmatched: 6


In [155]:
unmatched = dataset2_schedule_check[
    dataset2_schedule_check["_merge"] == "left_only"
][[
    "date",
    "Matchup",
    "team_1",
    "team_2",
    "match_key",
    "network",
    "viewership"
]]

unmatched

,date,Matchup,team_1,team_2,match_key,network,viewership
99,2026-04-08,Trail Blazers/Suns,POR,PHX,2026-04-08_PHX_POR,ESPN,1229000.0
116,2025-12-09,NBA Cup QF: Heat/Magic,MIA,ORL,2025-12-09_MIA_ORL,Prime Video,736000.0
118,2025-12-10,NBA Cup QF: Suns/Thunder,PHX,OKC,2025-12-10_OKC_PHX,Prime Video,1050000.0
119,2025-12-10,NBA Cup QF: Spurs/Lakers,SAS,LAL,2025-12-10_LAL_SAS,Prime Video,1460000.0
120,2025-12-13,NBA Cup SF: Knicks/Magic,NYK,ORL,2025-12-13_NYK_ORL,Prime Video,1330000.0
121,2025-12-13,NBA Cup SF: Spurs/Thunder,SAS,OKC,2025-12-13_OKC_SAS,Prime Video,2010000.0


In [156]:
dataset2_split.loc[
    dataset2_split["match_key"] == "2026-04-08_PHX_POR",
    ["Matchup", "team_1", "team_2", "match_key"]
]

,Matchup,team_1,team_2,match_key
100,Trail Blazers/Suns,POR,PHX,2026-04-08_PHX_POR


In [159]:
mask = (
    (dataset2_split["date"] == "2026-04-08") &
    (dataset2_split["Matchup"] == "Trail Blazers/Mavericks")
)

dataset2_split.loc[mask, "Matchup"] = "Mavericks/Suns"
dataset2_split.loc[mask, "Matchup_clean"] = "Mavericks/Suns"
dataset2_split.loc[mask, "team_1"] = "DAL"
dataset2_split.loc[mask, "team_2"] = "PHX"
dataset2_split.loc[mask, "match_key"] = "2026-04-08_DAL_PHX"

print(
    dataset2_split.loc[
        dataset2_split["date"] == "2026-04-08",
        ["Matchup", "team_1", "team_2", "match_key"]
    ]
)

             Matchup team_1 team_2           match_key
99   Hawks/Cavaliers    ATL    CLE  2026-04-08_ATL_CLE
100   Mavericks/Suns    DAL    PHX  2026-04-08_DAL_PHX


In [160]:
dataset2_schedule_check = dataset2_split.merge(
    master_schedule[
        ["match_key", "away_team", "home_team", "game_id",
         "season_type", "national_tv"]
    ],
    on="match_key",
    how="left",
    indicator=True
)

print("Dataset 2 rows:", len(dataset2_split))
print("Matched to master schedule:",
      (dataset2_schedule_check["_merge"] == "both").sum())
print("Unmatched:",
      (dataset2_schedule_check["_merge"] == "left_only").sum())

Dataset 2 rows: 202
Matched to master schedule: 197
Unmatched: 5


In [161]:
master_schedule[
    master_schedule["date"].isin([
        "2025-12-09",
        "2025-12-10",
        "2025-12-13"
    ])
][[
    "date",
    "match_key",
    "game_id",
    "season_type",
    "away_team",
    "home_team",
    "national_tv"
]].sort_values(["date", "match_key"])

,date,match_key,game_id,season_type,away_team,home_team,national_tv


In [162]:
nba_schedule[
    nba_schedule["tip_et"].astype(str).str.startswith(
        ("2025-12-09", "2025-12-10", "2025-12-13")
    )
][[
    "game_id",
    "season_type",
    "tip_et",
    "away_team",
    "home_team",
    "national_tv"
]]

,game_id,season_type,tip_et,away_team,home_team,national_tv
428,22501201,Regular Season,2025-12-09T00:00:00-05:00,(),(),Prime Video
429,22501203,Regular Season,2025-12-09T00:00:00-05:00,(),(),Prime Video
430,22501202,Regular Season,2025-12-10T00:00:00-05:00,(),(),Prime Video
431,22501204,Regular Season,2025-12-10T00:00:00-05:00,(),(),Prime Video
432,22501229,Regular Season,2025-12-13T00:00:00-05:00,(),(),Prime Video
433,22501230,Regular Season,2025-12-13T00:00:00-05:00,(),(),Prime Video


In [170]:
# NBA Cup playoff games
cup_games = pd.DataFrame({
    "date": pd.to_datetime([
        "2025-12-09",
        "2025-12-10",
        "2025-12-10",
        "2025-12-13",
        "2025-12-13"
    ]),
    "away_team": ["MIA", "PHX", "SAS", "NYK", "SAS"],
    "home_team": ["ORL", "OKC", "LAL", "ORL", "OKC"],
    "season_type": ["NBA Cup"] * 5,
    "national_tv": ["Prime Video"] * 5
})

cup_games["match_key"] = (
    cup_games["date"].dt.strftime("%Y-%m-%d")
    + "_"
    + cup_games[["away_team", "home_team"]]
        .apply(lambda x: "_".join(sorted(x)), axis=1)
)

cup_games

,date,away_team,home_team,season_type,national_tv,match_key
0,2025-12-09,MIA,ORL,NBA Cup,Prime Video,2025-12-09_MIA_ORL
1,2025-12-10,PHX,OKC,NBA Cup,Prime Video,2025-12-10_OKC_PHX
2,2025-12-10,SAS,LAL,NBA Cup,Prime Video,2025-12-10_LAL_SAS
3,2025-12-13,NYK,ORL,NBA Cup,Prime Video,2025-12-13_NYK_ORL
4,2025-12-13,SAS,OKC,NBA Cup,Prime Video,2025-12-13_OKC_SAS


In [171]:
#Adding Cup playoff games to Master Schedule
cup_games["game_id"] = pd.NA
cup_games["tip_et"] = pd.NaT

cup_games = cup_games[
    ["date", "match_key", "game_id", "season_type",
     "tip_et", "away_team", "home_team", "national_tv"]
]

master_schedule = pd.concat(
    [master_schedule, cup_games],
    ignore_index=True
)

print("master_schedule rows:", len(master_schedule))
print(
    master_schedule[
        master_schedule["season_type"] == "NBA Cup"
    ]
)

master_schedule rows: 1364
                     date           match_key game_id season_type tip_et  \
1359  2025-12-09 00:00:00  2025-12-09_MIA_ORL    <NA>     NBA Cup    NaT   
1360  2025-12-10 00:00:00  2025-12-10_OKC_PHX    <NA>     NBA Cup    NaT   
1361  2025-12-10 00:00:00  2025-12-10_LAL_SAS    <NA>     NBA Cup    NaT   
1362  2025-12-13 00:00:00  2025-12-13_NYK_ORL    <NA>     NBA Cup    NaT   
1363  2025-12-13 00:00:00  2025-12-13_OKC_SAS    <NA>     NBA Cup    NaT   

     away_team home_team  national_tv  
1359       MIA       ORL  Prime Video  
1360       PHX       OKC  Prime Video  
1361       SAS       LAL  Prime Video  
1362       NYK       ORL  Prime Video  
1363       SAS       OKC  Prime Video  


In [172]:
dataset2_schedule_check = dataset2_split.merge(
    master_schedule[
        ["match_key", "away_team", "home_team", "game_id",
         "season_type", "national_tv"]
    ],
    on="match_key",
    how="left",
    indicator=True
)

print("Dataset 2 rows:", len(dataset2_split))
print("Matched to master schedule:",
      (dataset2_schedule_check["_merge"] == "both").sum())
print("Unmatched:",
      (dataset2_schedule_check["_merge"] == "left_only").sum())

Dataset 2 rows: 202
Matched to master schedule: 202
Unmatched: 0


In [ ]:
#Finalizing Network Datasets

In [174]:
#ABC
abc_final = dataset2_split[
    dataset2_split["network"] == "ABC"
].copy()

abc_final["measurement_type"] = "average"
abc_final["measurement_source"] = "Nielsen Big Data"
abc_final["original_value_000s"] = abc_final["original_viewership"]

abc_final = abc_final[
    [
        "date",
        "team_1",
        "team_2",
        "network",
        "measurement_type",
        "measurement_source",
        "viewership",
        "original_value_000s",
        "measurement_scope"
    ]
]

print("ABC standardized rows:", len(abc_final))
print(abc_final.head())

ABC standardized rows: 23
        date team_1 team_2 network measurement_type measurement_source  \
0 2025-12-25    CLE    NYK     ABC          average   Nielsen Big Data   
1 2025-12-25    SAS    OKC     ABC          average   Nielsen Big Data   
2 2025-12-25    DAL    GSW     ABC          average   Nielsen Big Data   
3 2025-12-25    HOU    LAL     ABC          average   Nielsen Big Data   
4 2025-12-25    MIN    DEN     ABC          average   Nielsen Big Data   

   viewership       original_value_000s measurement_scope  
0   6400000.0  6400 (3818 + 2272 + 300)              game  
1   6711000.0        6711 (4343 + 2368)              game  
2   6107000.0          6107 (4199+1908)              game  
3   5349000.0         5349 (3450+ 1899)              game  
4   3610000.0          3610 (2166+1444)              game  


In [175]:
#NBC
nbc_final = dataset2_split[
    dataset2_split["network"] == "NBC"
].copy()

# Nielsen Big Data measurement
nbc_big_data = nbc_final[
    nbc_final["viewership"].notna()
].copy()

nbc_big_data["measurement_type"] = "average"
nbc_big_data["measurement_source"] = "Nielsen Big Data"
nbc_big_data["original_value_000s"] = nbc_big_data["original_viewership"]

nbc_big_data = nbc_big_data[
    [
        "date",
        "team_1",
        "team_2",
        "network",
        "measurement_type",
        "measurement_source",
        "viewership",
        "original_value_000s",
        "measurement_scope"
    ]
]

# Nielsen Big Data + Adobe Analytics measurement
nbc_combined = nbc_final[
    nbc_final["streaming_viewership"].notna()
].copy()

nbc_combined["measurement_type"] = "combined"
nbc_combined["measurement_source"] = (
    "Nielsen Big Data + Adobe Analytics"
)
nbc_combined["original_value_000s"] = nbc_combined["original_streaming"]
nbc_combined["viewership"] = nbc_combined["streaming_viewership"]

nbc_combined = nbc_combined[
    [
        "date",
        "team_1",
        "team_2",
        "network",
        "measurement_type",
        "measurement_source",
        "viewership",
        "original_value_000s",
        "measurement_scope"
    ]
]

nbc_final = pd.concat(
    [nbc_big_data, nbc_combined],
    ignore_index=True
)

print("NBC standardized rows:", len(nbc_final))
print("\nMeasurement source:")
print(nbc_final["measurement_source"].value_counts())
print("\nMeasurement type:")
print(nbc_final["measurement_type"].value_counts())

NBC standardized rows: 94

Measurement source:
Nielsen Big Data                      56
Nielsen Big Data + Adobe Analytics    38
Name: measurement_source, dtype: int64

Measurement type:
average     56
combined    38
Name: measurement_type, dtype: int64


In [176]:
#ESPN
espn_final = dataset2_split[
    dataset2_split["network"] == "ESPN"
].copy()

# Nielsen Big Data
espn_big_data = espn_final[
    espn_final["viewership"].notna()
].copy()

espn_big_data["measurement_type"] = "average"
espn_big_data["measurement_source"] = "Nielsen Big Data"
espn_big_data["original_value_000s"] = (
    espn_big_data["original_big_data"]
)

espn_big_data = espn_big_data[
    [
        "date",
        "team_1",
        "team_2",
        "network",
        "measurement_type",
        "measurement_source",
        "viewership",
        "original_value_000s",
        "measurement_scope"
    ]
]

# Nielsen Panel
espn_panel = espn_final[
    espn_final["panel_viewership"].notna()
].copy()

espn_panel["measurement_type"] = "average"
espn_panel["measurement_source"] = "Nielsen Panel"
espn_panel["original_value_000s"] = (
    espn_panel["original_panel"]
)
espn_panel["viewership"] = espn_panel["panel_viewership"]

espn_panel = espn_panel[
    [
        "date",
        "team_1",
        "team_2",
        "network",
        "measurement_type",
        "measurement_source",
        "viewership",
        "original_value_000s",
        "measurement_scope"
    ]
]

espn_final = pd.concat(
    [espn_big_data, espn_panel],
    ignore_index=True
)

print("ESPN standardized rows:", len(espn_final))
print("\nMeasurement source:")
print(espn_final["measurement_source"].value_counts())

ESPN standardized rows: 100

Measurement source:
Nielsen Big Data    51
Nielsen Panel       49
Name: measurement_source, dtype: int64


In [177]:
#PRIME VIDEO
prime_final = dataset2_split[
    dataset2_split["network"] == "Prime Video"
].copy()

prime_final = prime_final[
    prime_final["viewership"].notna()
].copy()

prime_final["measurement_type"] = "average"
prime_final["measurement_source"] = "Nielsen Big Data"
prime_final["original_value_000s"] = prime_final["original_viewership"]

prime_final = prime_final[
    [
        "date",
        "team_1",
        "team_2",
        "network",
        "measurement_type",
        "measurement_source",
        "viewership",
        "original_value_000s",
        "measurement_scope"
    ]
]

print("Prime Video standardized rows:", len(prime_final))
print(prime_final["measurement_scope"].value_counts())

Prime Video standardized rows: 24
game    24
Name: measurement_scope, dtype: int64


In [178]:
dataset2_final = pd.concat(
    [abc_final, nbc_final, espn_final, prime_final],
    ignore_index=True
)

print("Total standardized measurements:", len(dataset2_final))
print("\nBy network:")
print(dataset2_final["network"].value_counts())

print("\nBy measurement source:")
print(dataset2_final["measurement_source"].value_counts())

print("\nBy measurement scope:")
print(dataset2_final["measurement_scope"].value_counts())

print("\nMissing viewership:")
print(dataset2_final["viewership"].isna().sum())

Total standardized measurements: 241

By network:
ESPN           100
NBC             94
Prime Video     24
ABC             23
Name: network, dtype: int64

By measurement source:
Nielsen Big Data                      154
Nielsen Panel                          49
Nielsen Big Data + Adobe Analytics     38
Name: measurement_source, dtype: int64

By measurement scope:
game      187
window     54
Name: measurement_scope, dtype: int64

Missing viewership:
0


In [179]:
#ADDING CANONICAL MATCH KEY
dataset2_final["match_key"] = (
    dataset2_final["date"].dt.strftime("%Y-%m-%d")
    + "_"
    + dataset2_final[["team_1", "team_2"]]
        .apply(lambda x: "_".join(sorted(x)), axis=1)
)

print("Dataset 2 rows:", len(dataset2_final))
print("Unique match keys:", dataset2_final["match_key"].nunique())
print("\nSample:")
print(
    dataset2_final[
        ["date", "team_1", "team_2", "network",
         "measurement_source", "viewership",
         "measurement_scope", "match_key"]
    ].head(10)
)

Dataset 2 rows: 241
Unique match keys: 158

Sample:
        date team_1 team_2 network measurement_source  viewership  \
0 2025-12-25    CLE    NYK     ABC   Nielsen Big Data   6400000.0   
1 2025-12-25    SAS    OKC     ABC   Nielsen Big Data   6711000.0   
2 2025-12-25    DAL    GSW     ABC   Nielsen Big Data   6107000.0   
3 2025-12-25    HOU    LAL     ABC   Nielsen Big Data   5349000.0   
4 2025-12-25    MIN    DEN     ABC   Nielsen Big Data   3610000.0   
5 2026-01-24    NYK    PHI     ABC   Nielsen Big Data   2057000.0   
6 2026-01-24    LAL    DAL     ABC   Nielsen Big Data   2595000.0   
7 2026-01-31    DAL    HOU     ABC   Nielsen Big Data   1933000.0   
8 2026-02-07    HOU    OKC     ABC   Nielsen Big Data   1799000.0   
9 2026-02-07    GSW    LAL     ABC   Nielsen Big Data   2473000.0   

  measurement_scope           match_key  
0              game  2025-12-25_CLE_NYK  
1              game  2025-12-25_OKC_SAS  
2              game  2025-12-25_DAL_GSW  
3              game 

In [180]:
dataset2_joined = dataset2_final.merge(
    master_schedule[
        [
            "match_key",
            "game_id",
            "season_type",
            "tip_et",
            "away_team",
            "home_team",
            "national_tv"
        ]
    ],
    on="match_key",
    how="left",
    indicator=True
)

print("Dataset 2 rows:", len(dataset2_joined))
print(
    "Matched to master schedule:",
    (dataset2_joined["_merge"] == "both").sum()
)
print(
    "Unmatched:",
    (dataset2_joined["_merge"] == "left_only").sum()
)

Dataset 2 rows: 241
Matched to master schedule: 241
Unmatched: 0


In [181]:
#CLEANING DATASET 2 INTO FINAL STRUCTURE
dataset2_final = dataset2_joined.copy()

dataset2_final["source"] = "TV Media Blog"
dataset2_final["notes"] = dataset2_final["measurement_scope"].map({
    "game": "Game-specific audience measurement",
    "window": "Regional/multi-game audience window; same measurement applies to both games in the window"
})

dataset2_final = dataset2_final[
    [
        "date",
        "away_team",
        "home_team",
        "match_key",
        "national_tv",
        "viewership",
        "measurement_type",
        "measurement_source",
        "measurement_scope",
        "source",
        "notes",
        "game_id",
        "season_type"
    ]
]

print("Final Dataset 2 shape:", dataset2_final.shape)
print("\nColumns:")
print(dataset2_final.columns.tolist())
print("\nFirst 10 rows:")
print(dataset2_final.head(10))

Final Dataset 2 shape: (241, 13)

Columns:
['date', 'away_team', 'home_team', 'match_key', 'national_tv', 'viewership', 'measurement_type', 'measurement_source', 'measurement_scope', 'source', 'notes', 'game_id', 'season_type']

First 10 rows:
        date           away_team        home_team           match_key  \
0 2025-12-25     CLE (Cavaliers)     NYK (Knicks)  2025-12-25_CLE_NYK   
1 2025-12-25         SAS (Spurs)    OKC (Thunder)  2025-12-25_OKC_SAS   
2 2025-12-25     DAL (Mavericks)   GSW (Warriors)  2025-12-25_DAL_GSW   
3 2025-12-25       HOU (Rockets)     LAL (Lakers)  2025-12-25_HOU_LAL   
4 2025-12-25  MIN (Timberwolves)    DEN (Nuggets)  2025-12-25_DEN_MIN   
5 2026-01-24        NYK (Knicks)      PHI (76ers)  2026-01-24_NYK_PHI   
6 2026-01-24        LAL (Lakers)  DAL (Mavericks)  2026-01-24_DAL_LAL   
7 2026-01-31     DAL (Mavericks)    HOU (Rockets)  2026-01-31_DAL_HOU   
8 2026-02-07       HOU (Rockets)    OKC (Thunder)  2026-02-07_HOU_OKC   
9 2026-02-07      GSW (War

In [182]:
print("Season type:")
print(dataset2_final["season_type"].value_counts())

print("\nMeasurement type:")
print(dataset2_final["measurement_type"].value_counts())

print("\nMeasurement scope:")
print(dataset2_final["measurement_scope"].value_counts())

print("\nNetwork:")
print(dataset2_final["national_tv"].value_counts())

print("\nDuplicate measurements:")
print(
    dataset2_final.duplicated(
        subset=[
            "match_key",
            "measurement_source",
            "measurement_type",
            "viewership"
        ]
    ).sum()
)

Season type:
Regular Season    236
NBA Cup             5
Name: season_type, dtype: int64

Measurement type:
average     203
combined     38
Name: measurement_type, dtype: int64

Measurement scope:
game      187
window     54
Name: measurement_scope, dtype: int64

Network:
NBC | Peacock    85
ESPN             84
Prime Video      24
ABC              17
ABC | ESPN        5
Name: national_tv, dtype: int64

Duplicate measurements:
0


In [183]:
dataset2_final["source_url"] = (
    "https://tvmediablog.substack.com/p/2025-26-nba-viewership-database"
)

# Put source_url next to source for easier provenance tracking
dataset2_final = dataset2_final[
    [
        "date",
        "away_team",
        "home_team",
        "match_key",
        "national_tv",
        "viewership",
        "measurement_type",
        "measurement_source",
        "measurement_scope",
        "source",
        "source_url",
        "notes",
        "game_id",
        "season_type"
    ]
]

dataset2_final.to_csv(
    "/Users/drewbeal/NBA_Global_Media_Insights/Data/raw/"
    "tv_media_blog_viewership_2025_26.csv",
    index=False
)

print("Saved Dataset 2.")
print("Rows:", len(dataset2_final))
print("Columns:", len(dataset2_final.columns))

Saved Dataset 2.
Rows: 241
Columns: 14


In [ ]:
#COMPARING DATASET1 to DATASET2 to determine new observations/overlap

In [184]:
dataset1_keys = set(additional_viewership["match_key"])
dataset2_keys = set(dataset2_final["match_key"])

print("Dataset 1 unique games:", len(dataset1_keys))
print("Dataset 2 unique games:", len(dataset2_keys))
print("Games in both datasets:", len(dataset1_keys & dataset2_keys))
print("Games only in Dataset 1:", len(dataset1_keys - dataset2_keys))
print("Games only in Dataset 2:", len(dataset2_keys - dataset1_keys))

Dataset 1 unique games: 62
Dataset 2 unique games: 158
Games in both datasets: 60
Games only in Dataset 1: 2
Games only in Dataset 2: 98


In [185]:
#OBS in dataset1 but not dataset2
id="q6m31c"
dataset1_only = additional_viewership[
    ~additional_viewership["match_key"].isin(dataset2_keys)
][
    [
        "date",
        "away_team",
        "home_team",
        "match_key",
        "viewership",
        "measurement_type",
        "measurement_scope",
        "source"
    ]
]

dataset1_only

,date,away_team,home_team,match_key,viewership,measurement_type,measurement_scope,source
57,2026-02-03,CHI,MIL,2026-02-03_CHI_MIL,1050000,average,game,Sports Media Watch
65,2026-04-08,POR,SAS,2026-04-08_POR_SAS,1230000,average,game,Sports Media Watch


In [186]:
#obs in dataset 1 that are duplicated in dataset 2
id_cols = [
    "match_key",
    "viewership",
    "measurement_type"
]

dataset1_measurements = additional_viewership[id_cols].drop_duplicates()
dataset2_measurements = dataset2_final[id_cols].drop_duplicates()

measurement_overlap = dataset1_measurements.merge(
    dataset2_measurements,
    on=id_cols,
    how="inner"
)

print("Dataset 1 unique measurements:", len(dataset1_measurements))
print("Exact measurement matches:", len(measurement_overlap))
print(
    "Dataset 1 measurements not exactly duplicated in Dataset 2:",
    len(dataset1_measurements) - len(measurement_overlap)
)

Dataset 1 unique measurements: 67
Exact measurement matches: 16
Dataset 1 measurements not exactly duplicated in Dataset 2: 51


In [189]:
#Checking duplicates between dataset1 and 2
exact_duplicates = additional_viewership.merge(
    dataset2_final,
    on=["match_key", "viewership", "measurement_type"],
    how="inner",
    suffixes=("_dataset1", "_dataset2")
)

print("Exact duplicate rows:", len(exact_duplicates))

exact_duplicates[
    [
        "date_dataset1",
        "away_team_dataset1",
        "home_team_dataset1",
        "viewership",
        "measurement_type",
        "measurement_scope_dataset1",
        "source_dataset1",
        "measurement_source_dataset2"
    ]
]

Exact duplicate rows: 16


KeyError: "['measurement_source_dataset2'] not in index"

In [190]:
exact_duplicates[
    [
        "match_key",
        "viewership",
        "measurement_type",
        "measurement_scope_dataset1",
        "measurement_scope_dataset2",
        "source_dataset1",
        "measurement_source",
        "source_dataset2"
    ]


,match_key,viewership,measurement_type,measurement_scope_dataset1,measurement_scope_dataset2,source_dataset1,measurement_source,source_dataset2
0,2025-11-07_DEN_GSW,1060000,average,game,game,Sports Media Watch,Nielsen Big Data,TV Media Blog
1,2025-11-07_HOU_SAS,916000,average,game,game,Sports Media Watch,Nielsen Big Data,TV Media Blog
2,2025-12-23_DAL_DEN,2000000,combined,game,game,Sports Media Watch,Nielsen Big Data + Adobe Analytics,TV Media Blog
3,2025-12-23_HOU_LAC,1500000,combined,game,game,Sports Media Watch,Nielsen Big Data + Adobe Analytics,TV Media Blog
4,2025-12-29_CLE_SAS,1700000,combined,game,game,Sports Media Watch,Nielsen Big Data + Adobe Analytics,TV Media Blog
5,2025-12-29_DAL_POR,1200000,combined,game,game,Sports Media Watch,Nielsen Big Data + Adobe Analytics,TV Media Blog
6,2025-12-30_MEM_PHI,2200000,combined,game,game,Sports Media Watch,Nielsen Big Data + Adobe Analytics,TV Media Blog
7,2025-12-30_DET_LAL,2000000,combined,game,game,Sports Media Watch,Nielsen Big Data + Adobe Analytics,TV Media Blog
8,2026-01-15_HOU_OKC,982000,average,game,game,Sports Media Watch,Nielsen Big Data,TV Media Blog
9,2026-01-19_DAL_NYK,2200000,combined,game,game,Sports Media Watch,Nielsen Big Data + Adobe Analytics,TV Media Blog


In [192]:
#Building Combined Audience Table

# Identify exact Dataset 1 measurements already represented in Dataset 2
duplicate_keys = set(
    zip(
        exact_duplicates["match_key"],
        exact_duplicates["viewership"],
        exact_duplicates["measurement_type"]
    )
)

# Keep Dataset 1 observations that are NOT exact duplicates
dataset1_unique = additional_viewership[
    ~additional_viewership.apply(
        lambda row: (
            row["match_key"],
            row["viewership"],
            row["measurement_type"]
        ) in duplicate_keys,
        axis=1
    )
].copy()

# Combine the two datasets
master_audience = pd.concat(
    [dataset2_final, dataset1_unique],
    ignore_index=True,
    sort=False
)

print("Dataset 1 unique measurements retained:", len(dataset1_unique))
print("Dataset 2 measurements:", len(dataset2_final))
print("Master audience measurements:", len(master_audience))

print("\nDuplicate check:")
print(
    master_audience.duplicated(
        subset=["match_key", "viewership", "measurement_type"]
    ).sum()
)

Dataset 1 unique measurements retained: 51
Dataset 2 measurements: 241
Master audience measurements: 292

Duplicate check:
0


In [193]:
print("Master audience shape:", master_audience.shape)

print("\nBy source:")
print(master_audience["source"].value_counts())

print("\nBy measurement type:")
print(master_audience["measurement_type"].value_counts())

print("\nBy measurement scope:")
print(master_audience["measurement_scope"].value_counts())

print("\nBy measurement source:")
print(master_audience["measurement_source"].value_counts())

print("\nMissing values:")
print(master_audience.isna().sum())

Master audience shape: (292, 14)

By source:
TV Media Blog         241
Sports Media Watch     45
ESPN Press Room         5
NBC Sports              1
Name: source, dtype: int64

By measurement type:
average     244
combined     41
peak          7
Name: measurement_type, dtype: int64

By measurement scope:
game      238
window     54
Name: measurement_scope, dtype: int64

By measurement source:
Nielsen Big Data                      154
Nielsen Panel                          49
Nielsen Big Data + Adobe Analytics     38
Name: measurement_source, dtype: int64

Missing values:
date                   0
away_team              0
home_team              0
match_key              0
national_tv           26
viewership             0
measurement_type       0
measurement_source    51
measurement_scope      0
source                 0
source_url             0
notes                  0
game_id               58
season_type           51
dtype: int64


In [194]:
#How many Unique games and how many measurements per game
game_measurement_summary = (
    master_audience
    .groupby("match_key")
    .agg(
        measurements=("viewership", "count"),
        sources=("source", "nunique"),
        measurement_types=("measurement_type", "nunique"),
        scopes=("measurement_scope", "nunique")
    )
    .reset_index()
)

print("Unique games:", len(game_measurement_summary))
print("Games with 1 measurement:", 
      (game_measurement_summary["measurements"] == 1).sum())
print("Games with multiple measurements:", 
      (game_measurement_summary["measurements"] > 1).sum())

print("\nMeasurement count distribution:")
print(game_measurement_summary["measurements"].value_counts().sort_index())

Unique games: 160
Games with 1 measurement: 49
Games with multiple measurements: 111

Measurement count distribution:
1    49
2    90
3    21
Name: measurements, dtype: int64


In [199]:
raw_path = "NBA_Global_Media_Insights/Data/raw/"

master_audience.to_csv(
    raw_path + "master_audience.csv",
    index=False
)

master_schedule.to_csv(
    raw_path + "master_schedule.csv",
    index=False
)

dataset2_final.to_csv(
    raw_path + "tv_media_blog_viewership_2025_26.csv",
    index=False
)

print("Saved:")
print("- master_audience.csv:", len(master_audience), "rows")
print("- master_schedule.csv:", len(master_schedule), "rows")
print("- tv_media_blog_viewership_2025_26.csv:", len(dataset2_final), "rows")

Saved:
- master_audience.csv: 292 rows
- master_schedule.csv: 1364 rows
- tv_media_blog_viewership_2025_26.csv: 241 rows


In [201]:
import pandas as pd
import os

raw_path = "NBA_Global_Media_Insights/Data/raw/"

blog_files = {
    "ABC": "blog_abc.csv",
    "NBC": "blog_nbc.csv",
    "ESPN": "blog_espn.csv",
    "Prime Video": "blog_prime.csv"
}

# Get the 12 games with missing network
missing_keys = set(
    game_audience_clean.loc[
        game_audience_clean["network"].isna(),
        "match_key"
    ]
)

matches = []

for network, filename in blog_files.items():
    df = pd.read_csv(raw_path + filename)
    
    # Make sure match_key is available in the original file
    if "match_key" in df.columns:
        found = df[df["match_key"].isin(missing_keys)].copy()
        found["network_found"] = network
        matches.append(found)

blog_matches = pd.concat(matches, ignore_index=True)

blog_matches[
    ["match_key", "network_found"]
].sort_values("match_key")

NameError: name 'game_audience_clean' is not defined

In [ ]:
#TRANSITIONING TO SQL

In [196]:
%pip install duckdb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 24.6 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [197]:
import duckdb

print("DuckDB version:", duckdb.__version__)

DuckDB version: 1.5.6


In [198]:
import duckdb

con = duckdb.connect()

con.register("master_audience", master_audience)
con.register("master_schedule", master_schedule)

print("SQL tables registered:")
print(con.sql("SHOW TABLES"))

SQL tables registered:
┌─────────────────┐
│      name       │
│     varchar     │
├─────────────────┤
│ master_audience │
│ master_schedule │
└─────────────────┘

